In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 0 — Install · Imports · Seed · WHU dataset guard      ║
# ╚══════════════════════════════════════════════════════════════╝
# Run this cell first every session.
# It installs missing packages, sets the global random seed,
# detects GPU, and verifies the WHU dataset is attached.

import subprocess, sys

_REQUIRED = [
    "timm>=0.9", "tqdm", "Pillow", "scipy",
    "scikit-image", "opencv-python-headless",
]
for _pkg in _REQUIRED:
    _name = _pkg.split(">=")[0].replace("-", "_")
    try:
        __import__(_name)
    except ImportError:
        print(f"Installing {_pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _pkg])

# ── standard imports ─────────────────────────────────────────────
import math, os, glob, random, csv, time, shutil, hashlib
import threading, contextlib
from dataclasses import dataclass
from typing import Dict, Tuple, List
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from collections import OrderedDict, deque

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset
from PIL import Image
from tqdm.auto import tqdm
import timm

# ── reproducibility ───────────────────────────────────────────────
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[init] device = {device}")
if torch.cuda.is_available():
    print(f"[init] GPU = {torch.cuda.get_device_name(0)}"
          f"  VRAM = {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# ── WHU dataset guard ─────────────────────────────────────────────
WHU_ROOT = "/kaggle/input/datasets/sengulgs/whu-building-dataset/WHU"
if not os.path.isdir(WHU_ROOT):
    raise RuntimeError(
        f"WHU dataset not found at:\n  {WHU_ROOT}\n"
        "Fix: Kaggle editor -> Data tab -> + Add Data -> search 'whu-building-dataset'"
    )
print(f"[init] WHU dataset OK: {WHU_ROOT}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Config                                             ║
# ║                                                              ║
# ║  SMOKE_TEST = True  -> 5-minute sanity run (tiny data)        ║
# ║  SMOKE_TEST = False -> real training (WHU 1/8 labels)         ║
# ╚══════════════════════════════════════════════════════════════╝

SMOKE_TEST = True   # ← flip to False for real training

# ── smoke-test hard caps (ignored when SMOKE_TEST=False) ─────────
_SM_TRAIN   = 20    # total training images (labeled + unlabeled)
_SM_VAL     = 8     # validation images
_SM_PSEUDO  = 5     # max unlabeled batches in Stage 2
_SM_EPOCHS  = (1, 1, 1)   # (S1, S2, S3) epochs for smoke
_SM_WARMUP  = 1           # warmup epochs for smoke

# ── real-training defaults ────────────────────────────────────────
_RT_EPOCHS  = (15, 10, 5) # (S1, S2, S3)
_RT_WARMUP  = 3

def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

@dataclass
class Config:
    # ── dataset ──────────────────────────────────────────────────
    data_root: str   = "/kaggle/input/datasets/sengulgs/whu-building-dataset/WHU"
    img_size: int    = 512

    # ── label fraction (1/8 = 12.5% — primary paper result) ──────
    label_frac: float = 0.01 if SMOKE_TEST else 0.125

    # ── batch / workers ──────────────────────────────────────────
    batch_size: int   = 2 if SMOKE_TEST else 4
    num_workers: int  = 0 if SMOKE_TEST else 2

    # ── AMP + async gates ─────────────────────────────────────────
    use_amp: bool         = True
    use_async_gates: bool = not SMOKE_TEST      # CPU thread only in real mode
    async_warmup_batches: int = 5 if SMOKE_TEST else 50

    # ── epochs ───────────────────────────────────────────────────
    epochs_stage1: int  = _SM_EPOCHS[0] if SMOKE_TEST else _RT_EPOCHS[0]
    epochs_stage2: int  = _SM_EPOCHS[1] if SMOKE_TEST else _RT_EPOCHS[1]
    epochs_stage3: int  = _SM_EPOCHS[2] if SMOKE_TEST else _RT_EPOCHS[2]
    warmup_freeze: int  = _SM_WARMUP    if SMOKE_TEST else _RT_WARMUP

    # ── pseudo-label budget ───────────────────────────────────────
    # Stage 2 uses at most (pseudo_budget_N × labeled_count) unlabeled images.
    # Keeps Stage 2 proportional to supervision level; prevents 8000-batch runs.
    max_pseudo_batches: int = _SM_PSEUDO if SMOKE_TEST else 0  # 0 = use budget below
    pseudo_budget_N: int    = 3    # real mode: 3× labeled count as pseudo budget

    # ── smoke-test data caps ──────────────────────────────────────
    smoke_max_train: int = _SM_TRAIN
    smoke_max_val: int   = _SM_VAL

    # ── learning rates ───────────────────────────────────────────
    lr_backbone: float     = 1e-5
    lr_head: float         = 3e-4
    lr_loss_weights: float = 5e-5

    # ── pseudo-label thresholds (annealed each epoch) ─────────────
    thr_start: float = 0.65
    thr_end: float   = 0.50

    # ── loss weights ──────────────────────────────────────────────
    loss_w_in: float       = 1.0
    loss_w_out: float      = 1.5
    loss_dice_w: float     = 1.0
    boundary_weight: float = 2.5
    conf_loss_weight: float = 0.3

    # ── early stopping / saving ───────────────────────────────────
    patience: int     = 3  if SMOKE_TEST else 12
    viz_every: int    = 1  if SMOKE_TEST else 2
    pred_thr: float   = 0.5
    preview_every: int = 0
    preview_samples: int = 2

    # ── paths ─────────────────────────────────────────────────────
    save_path: str       = "/kaggle/working/best_model.pt"
    results_dir: str     = "/kaggle/working/results"
    checkpoint_path: str = "/kaggle/working/training_checkpoint.pt"
    persist_dir: str     = "/kaggle/working/persist"

    # ── EDL ──────────────────────────────────────────────────────
    edl_loss_weight: float    = 0.5
    edl_annealing_epochs: int = 2 if SMOKE_TEST else 10

    # ── CAM-Guided CLAAM v2 ───────────────────────────────────────
    claam_loss_weight: float      = 0.4
    claam_cam_weight: float       = 0.5
    claam_contrast_weight: float  = 0.5
    claam_entropy_weight: float   = 0.05
    claam_high_margin: float      = 0.7
    claam_low_margin: float       = 0.3
    claam_diversity_weight: float = 0.0
    claam_thr_start: float        = 0.55
    claam_thr_end: float          = 0.40
    cam_freeze_epochs: int        = 0 if SMOKE_TEST else 5

    # ── CLAC ──────────────────────────────────────────────────────
    clac_loss_weight: float = 0.2

    # ── TVR ───────────────────────────────────────────────────────
    tvr_window: int         = 5
    tvr_accept_ratio: float = 0.6

    # ── GOP ───────────────────────────────────────────────────────
    gop_rectilinear_thr: float = 0.45

    # ── CAFCG ─────────────────────────────────────────────────────
    cafcg_disagreement_thr: float = 0.3
    cafcg_keep_q: float           = 0.70

    # ── SDR ───────────────────────────────────────────────────────
    sdr_loss_weight: float = 0.3
    sdr_max_dist: float    = 50.0

    # ── pseudo misc ───────────────────────────────────────────────
    pseudo_ramp_start: float     = 0.1
    pseudo_freeze_backbone: bool = True
    pseudo_edge_erode_px: int    = 5

    # ── cross-dataset ─────────────────────────────────────────────
    inria_root: str             = "/kaggle/input/inria-building/AerialImageDataset"
    cross_dataset_img_size: int = 512

    log_every_n_batches: int = 0

cfg = Config()
os.makedirs(cfg.persist_dir, exist_ok=True)
os.makedirs(cfg.results_dir, exist_ok=True)

print(f"[config] SMOKE_TEST = {SMOKE_TEST}")
print(f"[config] label_frac = {cfg.label_frac}  "
      f"epochs S1/S2/S3 = {cfg.epochs_stage1}/{cfg.epochs_stage2}/{cfg.epochs_stage3}")
print(f"[config] batch = {cfg.batch_size}  "
      f"async_gates = {cfg.use_async_gates}  "
      f"max_pseudo_batches = {cfg.max_pseudo_batches}")
print(f"[config] persist_dir = {cfg.persist_dir}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Image utilities                                    ║
# ║  rgb_to_gray · hh_wavelet_channel · EdgeBranch               ║
# ╚══════════════════════════════════════════════════════════════╝

def rgb_to_gray(x: torch.Tensor) -> torch.Tensor:
    """Luminance-weighted grayscale  [B,3,H,W] -> [B,1,H,W]."""
    r, g, b = x[:, 0:1], x[:, 1:2], x[:, 2:3]
    return 0.2989 * r + 0.5870 * g + 0.1140 * b


def hh_wavelet_channel(x: torch.Tensor) -> torch.Tensor:
    """Haar HH (diagonal edge) channel  [B,3,H,W] -> [B,1,H,W].
    Highlights fine-grained boundary textures for the edge branch."""
    gray   = rgb_to_gray(x)
    kernel = torch.tensor([[1., -1.], [-1., 1.]],
                          device=gray.device).view(1, 1, 2, 2)
    hh = F.conv2d(gray, kernel, stride=1, padding=1)
    return hh[:, :, :x.shape[-2], :x.shape[-1]]


class EdgeBranch(nn.Module):
    """Lightweight 3-layer CNN that produces a soft edge probability map.
    Input: 4 channels (RGB + HH wavelet).  Output: 1-channel logit map."""
    def __init__(self, in_ch: int = 4):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32,    32, 3, padding=1)
        self.conv3 = nn.Conv2d(32,     1, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        return self.conv3(x)     # raw logit — caller applies sigmoid if needed

print("[cell2] image utilities defined")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 3 — DABLCNet architecture                              ║
# ║  Backbone : ViT-B/16 (timm, 512-res, 32×32 patch grid)       ║
# ║  Neck     : MultiLayerFusion (4 ViT layers) + ASPP           ║
# ║  Decoder  : EdgeGuidedDecoder + SpatialEncoder skips         ║
# ║  Heads    : logits · EDL (Dirichlet) · CLAAM · SDR           ║
# ╚══════════════════════════════════════════════════════════════╝

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

class SpatialEncoder(nn.Module):
    """Lightweight CNN for multi-scale spatial features -> decoder skip connections.
    Gives the decoder high-res boundary info that ViT's 32×32 bottleneck loses.
    Only ~160K params — negligible vs ViT's 86M."""
    def __init__(self):
        super().__init__()
        # 512 -> 256, 32ch
        self.layer1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True))
        # 256 -> 128, 64ch
        self.layer2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True))

    def forward(self, x):
        s1 = self.layer1(x)   # [B, 32, H/2, W/2]  = 256
        s2 = self.layer2(s1)  # [B, 64, H/4, W/4]  = 128
        return s1, s2


class ViTAttentionExtractor:
    """Capture ViT CLS-to-patch attention from selected transformer blocks."""
    def __init__(self, vit_encoder, layers=(2, 5, 8, 11), detach=True):
        self.layers = list(layers)
        self.detach = detach
        self._maps = {}
        self._hooks = []
        self._register(vit_encoder)

    def _register(self, vit_encoder):
        for idx in self.layers:
            block = vit_encoder.blocks[idx]
            hook = block.attn.register_forward_hook(self._make_hook(idx))
            self._hooks.append(hook)

    def _make_hook(self, layer_idx):
        def hook(module, input, output):
            x = input[0]
            B, N, C = x.shape
            qkv = module.qkv(x).reshape(B, N, 3, module.num_heads, C // module.num_heads)
            qkv = qkv.permute(2, 0, 3, 1, 4)
            q, k = qkv[0], qkv[1]
            scale = getattr(module, "scale", q.shape[-1] ** -0.5)
            attn = (q @ k.transpose(-2, -1)) * scale
            attn = attn.softmax(dim=-1)
            self._maps[layer_idx] = attn.detach() if self.detach else attn
        return hook

    def clear(self):
        self._maps = {}

    def get_spatial_maps(self, patch_h, patch_w):
        spatial = []
        for idx in self.layers:
            if idx not in self._maps:
                raise RuntimeError(f"Attention map for ViT layer {idx} was not captured")
            attn = self._maps[idx]
            cls_attn = attn[:, :, 0, 1:]
            expected = patch_h * patch_w
            if cls_attn.shape[-1] != expected:
                grid = int(math.sqrt(cls_attn.shape[-1]))
                if grid * grid != cls_attn.shape[-1]:
                    raise ValueError(f"Cannot reshape {cls_attn.shape[-1]} patch tokens into a grid")
                patch_h = patch_w = grid
            spatial.append(cls_attn.reshape(cls_attn.shape[0], cls_attn.shape[1], patch_h, patch_w))
        return spatial

    def remove_hooks(self):
        for hook in self._hooks:
            hook.remove()
        self._hooks.clear()


class ViTEncoder(nn.Module):
    """512-resolution ViT: 512/16 = 32x32 tokens.
    Returns the same multi-layer feature maps as Phase-2, and stores
    attention maps for CAM-Guided CLAAM.
    """
    def __init__(self):
        super().__init__()
        if timm is None:
            raise RuntimeError("timm not available.")
        self.vit = timm.create_model(
            "vit_base_patch16_224", pretrained=True,
            num_classes=0, img_size=512)
        try:
            self.vit.set_grad_checkpointing(enable=True)
        except AttributeError:
            pass
        self.patch_size = 16
        self.embed_dim = 768
        self.hook_layers = [2, 5, 8, 11]
        self._features = {}
        self._last_attn_maps = None
        self._register_hooks()
        self.attn_extractor = ViTAttentionExtractor(self.vit, layers=self.hook_layers)

    def _register_hooks(self):
        for idx in self.hook_layers:
            block = self.vit.blocks[idx]
            block.register_forward_hook(self._make_hook(idx))

    def _make_hook(self, idx):
        def hook_fn(module, input, output):
            self._features[idx] = output
        return hook_fn

    def forward(self, x: torch.Tensor):
        mean = IMAGENET_MEAN.to(x.device)
        std = IMAGENET_STD.to(x.device)
        x = (x - mean) / std

        self._features = {}
        self.attn_extractor.clear()
        _ = self.vit.forward_features(x)

        b = x.shape[0]
        grid_h = x.shape[-2] // self.patch_size
        grid_w = x.shape[-1] // self.patch_size

        multi_feats = []
        for idx in self.hook_layers:
            feat = self._features[idx]
            if feat.shape[1] == grid_h * grid_w + 1:
                feat = feat[:, 1:, :]
            feat_map = feat.transpose(1, 2).reshape(b, self.embed_dim, grid_h, grid_w)
            multi_feats.append(feat_map)

        self._last_attn_maps = self.attn_extractor.get_spatial_maps(grid_h, grid_w)
        return multi_feats

    def get_attention_maps(self):
        if self._last_attn_maps is None:
            raise RuntimeError("Run the ViT encoder before requesting attention maps")
        return self._last_attn_maps


class ASPP(nn.Module):
    """Atrous Spatial Pyramid Pooling (DeepLabV3+ style).
    Captures multi-scale context at the bottleneck — buildings come in many sizes.
    Parallel dilated convolutions at rates 6, 12, 18 + global average pooling."""
    def __init__(self, in_ch, out_ch=256):
        super().__init__()
        self.conv1x1 = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
        self.atrous6 = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=6, dilation=6, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
        self.atrous12 = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=12, dilation=12, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
        self.atrous18 = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=18, dilation=18, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
        self.gap = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.ReLU(inplace=True))
        self.project = nn.Sequential(
            nn.Conv2d(out_ch * 5, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Dropout(0.1))

    def forward(self, x):
        h, w = x.shape[-2:]
        f1 = self.conv1x1(x)
        f2 = self.atrous6(x)
        f3 = self.atrous12(x)
        f4 = self.atrous18(x)
        f5 = self.gap(x)
        f5 = F.interpolate(f5, size=(h, w), mode="bilinear", align_corners=False)
        out = torch.cat([f1, f2, f3, f4, f5], dim=1)
        return self.project(out)


class MultiLayerFusion(nn.Module):
    """Fuse 4 ViT layers (each 768-ch @ 32×32) into a single feature map.
    Projects each layer to 256-ch, concatenates, then compresses to out_ch."""
    def __init__(self, embed_dim=768, out_ch=768):
        super().__init__()
        self.proj3  = nn.Sequential(nn.Conv2d(embed_dim, 256, 1), nn.BatchNorm2d(256), nn.ReLU(True))
        self.proj6  = nn.Sequential(nn.Conv2d(embed_dim, 256, 1), nn.BatchNorm2d(256), nn.ReLU(True))
        self.proj9  = nn.Sequential(nn.Conv2d(embed_dim, 256, 1), nn.BatchNorm2d(256), nn.ReLU(True))
        self.proj12 = nn.Sequential(nn.Conv2d(embed_dim, 256, 1), nn.BatchNorm2d(256), nn.ReLU(True))
        self.fuse = nn.Sequential(
            nn.Conv2d(256 * 4, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))

    def forward(self, multi_feats):
        p3  = self.proj3(multi_feats[0])
        p6  = self.proj6(multi_feats[1])
        p9  = self.proj9(multi_feats[2])
        p12 = self.proj12(multi_feats[3])
        return self.fuse(torch.cat([p3, p6, p9, p12], dim=1))


class EdgeGuidedDecoder(nn.Module):
    """Decoder for 32×32 bottleneck -> 512. All learned upsampling — no bilinear blur.
    32->64->128(+skip@128)->256(+skip@256)->512 with edge fusion."""
    def __init__(self, in_ch: int = 256):
        super().__init__()
        self.up1 = nn.Sequential(
            nn.ConvTranspose2d(in_ch, 256, 2, stride=2),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True))
        self.up2 = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 2, stride=2),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True))
        self.skip_fuse2 = nn.Sequential(
            nn.Conv2d(192, 128, 3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True))
        self.up3 = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 2, stride=2),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True))
        self.skip_fuse3 = nn.Sequential(
            nn.Conv2d(96, 64, 3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True))
        self.up4 = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 2, stride=2),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True))
        self.refine = nn.Sequential(
            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True))
        self.fuse = nn.Sequential(
            nn.Conv2d(32 + 1, 32, 3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True))
        self.out_conv = nn.Conv2d(32, 1, 1)

    def _decode(self, feat, out_size, skips=None):
        """Shared decode path: 32->64->128->256->512."""
        x = self.up1(feat)
        x = self.up2(x)
        if skips is not None:
            s2 = F.interpolate(skips[1], size=x.shape[-2:],
                               mode="bilinear", align_corners=False)
            x = self.skip_fuse2(torch.cat([x, s2], dim=1))
        x = self.up3(x)
        if skips is not None:
            s1 = F.interpolate(skips[0], size=x.shape[-2:],
                               mode="bilinear", align_corners=False)
            x = self.skip_fuse3(torch.cat([x, s1], dim=1))
        x = self.up4(x)
        if x.shape[-2] != out_size[0] or x.shape[-1] != out_size[1]:
            x = F.interpolate(x, size=out_size, mode="nearest")
        x = self.refine(x)
        return x

    def forward(self, feat: torch.Tensor, edge: torch.Tensor,
                out_size: Tuple[int, int], skips=None) -> torch.Tensor:
        x = self._decode(feat, out_size, skips)
        edge_resized = F.interpolate(edge, size=out_size, mode="nearest")
        x = torch.cat([x, edge_resized], dim=1)
        x = self.fuse(x)
        return self.out_conv(x)

    def get_features(self, feat: torch.Tensor, out_size: Tuple[int, int],
                     skips=None) -> torch.Tensor:
        """Return 32-ch feature map for confidence head (no edge fusion)."""
        return self._decode(feat, out_size, skips)


class ConfidenceHead(nn.Module):
    """Original sigmoid confidence head (baseline / fallback when EDL disabled)."""
    def __init__(self, in_ch: int = 32):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 16, 3, padding=1)
        self.conv3 = nn.Conv2d(16, 1, 1)
        self.log_temp = nn.Parameter(torch.zeros(1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.conv3(x)
        temp = torch.exp(self.log_temp) + 1e-6
        return torch.sigmoid(x / temp)


# ═══════════════════════════════════════════════════════════════════
#  CONTRIBUTION 1: Evidential Dirichlet Confidence Head (EDL-Conf)
# ═══════════════════════════════════════════════════════════════════

class EvidentialHead(nn.Module):
    """Outputs Dirichlet concentration params α = evidence + 1 per class.
    Epistemic uncertainty u = K/S where S = Σα, K = num_classes."""
    def __init__(self, in_ch: int = 32, num_classes: int = 2):
        super().__init__()
        self.K = num_classes
        self.conv1 = nn.Conv2d(in_ch, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 16, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(16)
        self.conv3 = nn.Conv2d(16, num_classes, 1)

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        evidence = F.softplus(self.conv3(x))
        alpha = evidence + 1.0
        S = alpha.sum(dim=1, keepdim=True)
        belief = evidence / S
        uncertainty = float(self.K) / S
        confidence = 1.0 - uncertainty
        return {
            "alpha": alpha,
            "belief": belief,
            "uncertainty": uncertainty,
            "confidence": confidence.clamp(0, 1),
            "strength": S,
        }


def kl_dirichlet(alpha: torch.Tensor, num_classes: int = 2) -> torch.Tensor:
    """KL divergence between Dir(α) and Dir(1,...,1)."""
    ones = torch.ones_like(alpha)
    S_alpha = alpha.sum(dim=1, keepdim=True)
    S_ones = ones.sum(dim=1, keepdim=True)
    kl = (torch.lgamma(S_alpha) - torch.lgamma(S_ones)
          - (torch.lgamma(alpha) - torch.lgamma(ones)).sum(dim=1, keepdim=True)
          + ((alpha - ones) * (torch.digamma(alpha) - torch.digamma(S_alpha))).sum(dim=1, keepdim=True))
    return kl.mean()


def edl_loss(alpha, target, epoch, total_epochs, annealing_epochs=10):
    """EDL loss = Bayes risk + annealed KL."""
    target_1h = torch.cat([1.0 - target, target], dim=1)
    S = alpha.sum(dim=1, keepdim=True)
    bayes_risk = (target_1h * (torch.digamma(S) - torch.digamma(alpha))).sum(dim=1)
    annealing_coef = min(1.0, epoch / max(1, annealing_epochs))
    alpha_tilde = target_1h + (1.0 - target_1h) * (alpha - 1.0) + 1.0
    kl = kl_dirichlet(alpha_tilde, num_classes=2)
    return bayes_risk.mean() + annealing_coef * kl


# ═══════════════════════════════════════════════════════════════════
#  CONTRIBUTION 2: Cross-Layer Attention Agreement Map (CLAAM / MLAA)
# ═══════════════════════════════════════════════════════════════════

class CAMGuidedCLAAM(nn.Module):
    """v2: removes per-image min-max norm, adds per-layer projection
    and global tanh-gain. Output keys are unchanged."""
    def __init__(self, embed_dim: int = 768, num_heads: int = 12,
                 num_layers: int = 4, dropout: float = 0.1):
        super().__init__()
        self.num_heads = num_heads
        self.num_layers = num_layers

        self.cam_generator = nn.Sequential(
            nn.Conv2d(embed_dim, embed_dim // 4, kernel_size=1, bias=False),
            nn.BatchNorm2d(embed_dim // 4),
            nn.GELU(),
            nn.Dropout2d(dropout),
            nn.Conv2d(embed_dim // 4, embed_dim // 16, kernel_size=3,
                      padding=1, bias=False),
            nn.BatchNorm2d(embed_dim // 16),
            nn.GELU(),
            nn.Conv2d(embed_dim // 16, 1, kernel_size=1),
        )

        # NEW v2: per-layer projection forces specialization between layers
        # whose ViT features are nearly identical (DeepViT attention collapse).
        self.layer_proj = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(num_heads, num_heads, kernel_size=1,
                          groups=num_heads, bias=False),
                nn.BatchNorm2d(num_heads),
                nn.GELU(),
            ) for _ in range(num_layers)
        ])

        self.head_weights = nn.Parameter(torch.ones(num_heads) / num_heads)
        self.layer_weights = nn.Parameter(torch.ones(num_layers) / num_layers)
        self.temperature = nn.Parameter(torch.tensor(1.0))

        # Refine WITHOUT trailing Sigmoid; output gets gain-tanh below.
        self.refine = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(16, 1, kernel_size=3, padding=1),
        )

        # NEW v2: global gain+bias REPLACE per-image min-max normalization.
        self.out_gain = nn.Parameter(torch.tensor(2.0))
        self.out_bias = nn.Parameter(torch.tensor(0.0))

    def forward(self, attn_maps_per_layer, feature_map, out_size):
        cam_logits = self.cam_generator(feature_map)
        cam = torch.sigmoid(cam_logits)

        projected = [self.layer_proj[i](a)
                     for i, a in enumerate(attn_maps_per_layer)]
        attn_stack = torch.stack(projected, dim=1)
        attn_cam = attn_stack * cam.unsqueeze(1)

        head_w = self.head_weights.softmax(dim=0)
        layer_w = self.layer_weights.softmax(dim=0)
        attn_hw = attn_cam * head_w.view(1, 1, -1, 1, 1)
        attn_lw = attn_hw * layer_w.view(1, -1, 1, 1, 1)

        layer_consensus = attn_lw.mean(dim=2)
        layer_std = layer_consensus.std(dim=1)
        layer_mean = layer_consensus.mean(dim=1)

        temp = self.temperature.clamp(min=0.1)
        raw_agreement = layer_mean / (1.0 + layer_std / temp)
        refined = self.refine(raw_agreement.unsqueeze(1))

        # KEY CHANGE: global tanh-gain instead of per-image min-max normalization
        agreement = 0.5 * (torch.tanh(self.out_gain * refined + self.out_bias) + 1.0)

        variance = layer_std.unsqueeze(1)
        mean_pred = layer_mean.unsqueeze(1)

        if agreement.shape[-2:] != out_size:
            agreement = F.interpolate(agreement, size=out_size, mode="bilinear", align_corners=False)
            variance = F.interpolate(variance, size=out_size, mode="bilinear", align_corners=False)
            mean_pred = F.interpolate(mean_pred, size=out_size, mode="bilinear", align_corners=False)
            cam = F.interpolate(cam, size=out_size, mode="bilinear", align_corners=False)
            cam_logits = F.interpolate(cam_logits, size=out_size, mode="bilinear", align_corners=False)

        return {
            "agreement": agreement,
            "variance": variance,
            "cam": cam,
            "cam_logits": cam_logits,
            "mean_pred": mean_pred,
            "head_weights": self.head_weights.softmax(dim=0),
            "layer_weights": self.layer_weights.softmax(dim=0),
        }

CLAAM = CAMGuidedCLAAM


def claam_consistency_loss(layer_logits, target):
    """Force ALL ViT layers to agree with GT."""
    loss = torch.tensor(0.0, device=target.device)
    for logit in layer_logits:
        loss = loss + F.binary_cross_entropy_with_logits(logit, target)
    return loss / len(layer_logits)


# ═══════════════════════════════════════════════════════════════════
#  IDEA 2: Cross-Layer Attention Consistency Loss (CLAC-Loss)
# ═══════════════════════════════════════════════════════════════════

def clac_loss(multi_feats: List[torch.Tensor]) -> torch.Tensor:
    """Penalizes spatial activation disagreement between adjacent ViT layers."""
    loss = torch.tensor(0.0, device=multi_feats[0].device)
    for i in range(len(multi_feats) - 1):
        act_i = multi_feats[i].mean(dim=1, keepdim=True)
        act_j = multi_feats[i + 1].mean(dim=1, keepdim=True)
        act_i = (act_i - act_i.amin(dim=(2, 3), keepdim=True)) / \
                (act_i.amax(dim=(2, 3), keepdim=True) - act_i.amin(dim=(2, 3), keepdim=True) + 1e-6)
        act_j = (act_j - act_j.amin(dim=(2, 3), keepdim=True)) / \
                (act_j.amax(dim=(2, 3), keepdim=True) - act_j.amin(dim=(2, 3), keepdim=True) + 1e-6)
        loss = loss + F.mse_loss(act_i, act_j)
    return loss / (len(multi_feats) - 1)


# ═══════════════════════════════════════════════════════════════════
#  IDEA 4: Gradient Orientation Prior (GOP)
# ═══════════════════════════════════════════════════════════════════

def gradient_orientation_gate(probs: torch.Tensor, thr: float = 0.5,
                              rectilinear_thr: float = 0.45) -> torch.Tensor:
    """Reject pseudo-label components whose boundary gradient orientations
    deviate from rectilinear (0°/90°) angles."""
    import cv2
    B = probs.shape[0]
    gate = torch.ones_like(probs, dtype=torch.bool)

    for b in range(B):
        mask_np = (probs[b, 0].detach().cpu().numpy() > thr).astype(np.uint8) * 255
        if mask_np.sum() == 0:
            continue

        # AMP fix: probs may be float16 under autocast; cv2.Sobel needs float32/64.
        prob_np = probs[b, 0].detach().float().cpu().numpy()
        gx = cv2.Sobel(prob_np, cv2.CV_64F, 1, 0, ksize=3)
        gy = cv2.Sobel(prob_np, cv2.CV_64F, 0, 1, ksize=3)
        mag = np.sqrt(gx**2 + gy**2)
        angle = np.degrees(np.arctan2(gy, gx)) % 180

        contours, _ = cv2.findContours(mask_np, cv2.RETR_EXTERNAL,
                                        cv2.CHAIN_APPROX_SIMPLE)
        reject_mask = np.zeros_like(mask_np, dtype=np.uint8)

        for cnt in contours:
            area = cv2.contourArea(cnt)
            if area < 30:
                continue
            comp_mask = np.zeros_like(mask_np)
            cv2.drawContours(comp_mask, [cnt], -1, 255, thickness=cv2.FILLED)
            kernel = np.ones((3, 3), np.uint8)
            dilated = cv2.dilate(comp_mask, kernel, iterations=2)
            eroded = cv2.erode(comp_mask, kernel, iterations=1)
            boundary = ((dilated > 0) & ~(eroded > 0))

            boundary_mag = mag[boundary]
            boundary_angle = angle[boundary]
            if boundary_mag.size == 0:
                continue
            strong = boundary_mag > (boundary_mag.mean() + 1e-8)

            if strong.sum() < 5:
                continue

            angles_strong = boundary_angle[strong]
            rectilinear = (
                (angles_strong < 15) | (angles_strong > 165) |
                ((angles_strong > 75) & (angles_strong < 105))
            )
            rect_frac = rectilinear.sum() / len(angles_strong)

            if rect_frac < rectilinear_thr:
                cv2.drawContours(reject_mask, [cnt], -1, 255, thickness=cv2.FILLED)

        if reject_mask.any():
            reject_t = torch.from_numpy(reject_mask > 0).to(probs.device)
            gate[b, 0] = gate[b, 0] & (~reject_t)

    return gate


# ═══════════════════════════════════════════════════════════════════
#  IDEA 5a: Confidence-Aware Frequency Consistency Gate (CAFCG)
# ═══════════════════════════════════════════════════════════════════

def cafcg_gate(probs: torch.Tensor, disagreement_thr: float = None,
               keep_quantile: float = None) -> torch.Tensor:
    """CAFCG v2: ratio-based detector + adaptive per-image quantile threshold.
    The legacy `disagreement_thr` argument is kept for back-compat but ignored;
    use `keep_quantile` (or cfg.cafcg_keep_q) to control selectivity."""
    if keep_quantile is None:
        keep_quantile = getattr(cfg, "cafcg_keep_q", 0.70)
    B, C, H, W = probs.shape
    ksize, sigma = 9, 2.0
    x = torch.arange(ksize, device=probs.device).float() - ksize // 2
    g1d = torch.exp(-x ** 2 / (2 * sigma ** 2))
    g1d = g1d / g1d.sum()
    g2d = (g1d.unsqueeze(1) * g1d.unsqueeze(0)).view(1, 1, ksize, ksize)
    low_freq = F.conv2d(probs, g2d, padding=ksize // 2)
    high_freq = (probs - low_freq).abs()
    ratio = high_freq / (low_freq.clamp_min(0.05))
    flat = ratio.flatten(2)
    thr = torch.quantile(flat, keep_quantile, dim=2, keepdim=True).unsqueeze(-1)
    return ratio <= thr


class SignedDistanceHead(nn.Module):
    """Predicts normalized signed distance field (SDF) from decoder features.
    Output range: tanh -> [-1, +1] where +1 = deep interior, -1 = far outside,
    0 = on building boundary."""
    def __init__(self, in_ch: int = 32):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, 16, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(16, 1, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.bn1(self.conv1(x)))
        return torch.tanh(self.conv2(x))  # [-1, +1]


def compute_sdf_target(mask: torch.Tensor, max_dist: float = 50.0) -> torch.Tensor:
    """Compute normalized signed distance field from binary GT mask.
    Args:
        mask: [B, 1, H, W] binary building mask
        max_dist: clip and normalize distances to [-1, +1]
    Returns:
        sdf: [B, 1, H, W] where +1 = deep inside, -1 = far outside, 0 = boundary
    """
    from scipy.ndimage import distance_transform_edt
    B = mask.shape[0]
    sdf_batch = torch.zeros_like(mask)
    for b in range(B):
        m = mask[b, 0].cpu().numpy().astype(np.float64)
        if m.sum() > 0:
            dist_in = distance_transform_edt(m)
        else:
            dist_in = np.zeros_like(m)
        if (1 - m).sum() > 0:
            dist_out = distance_transform_edt(1 - m)
        else:
            dist_out = np.zeros_like(m)
        sdf = dist_in - dist_out  # + inside, - outside, 0 on boundary
        sdf = np.clip(sdf / max_dist, -1.0, 1.0)
        sdf_batch[b, 0] = torch.from_numpy(sdf).float()
    return sdf_batch.to(mask.device)


# ═══════════════════════════════════════════════════════════════════
#  Updated DABLCNet: integrates EDL, CLAAM, SDR, and all novel modules
# ═══════════════════════════════════════════════════════════════════

class DABLCNet(nn.Module):
    """DABLCNet + Phase-2 contributions with CAM-Guided CLAAM swapped in."""
    def __init__(self):
        super().__init__()

        # Core architecture
        self.encoder = ViTEncoder()
        self.layer_fuse = MultiLayerFusion(768, out_ch=768)
        self.aspp = ASPP(in_ch=768, out_ch=256)
        self.spatial_enc = SpatialEncoder()
        self.edge_branch = EdgeBranch(in_ch=4)
        self.decoder = EdgeGuidedDecoder(in_ch=256)

        # Contribution 1: Evidential Dirichlet head
        self.edl_head = EvidentialHead(in_ch=32, num_classes=2)

        # Contribution 2: CAM-Guided CLAAM
        self.claam = CAMGuidedCLAAM(embed_dim=768, num_heads=12, num_layers=4)

        # SDR: Signed Distance Regression head
        self.sdr_head = SignedDistanceHead(in_ch=32)

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        hh = hh_wavelet_channel(x)
        edge_in = torch.cat([x, hh], dim=1)
        edge_map = self.edge_branch(edge_in)

        multi_feats = self.encoder(x)
        attn_maps = self.encoder.get_attention_maps()

        fused = self.layer_fuse(multi_feats)
        bottleneck = self.aspp(fused)
        skips = self.spatial_enc(x)
        logits = self.decoder(bottleneck, edge_map,
                              out_size=x.shape[-2:], skips=skips)

        dec_feat = self.decoder.get_features(bottleneck,
                                              out_size=x.shape[-2:], skips=skips)

        edl_out = self.edl_head(dec_feat)
        conf_map = edl_out["confidence"]

        claam_out = self.claam(attn_maps, multi_feats[-1], out_size=x.shape[-2:])
        sdf_pred = self.sdr_head(dec_feat)

        return {
            "logits": logits,
            "edge": edge_map,
            "conf": conf_map,
            "edl": edl_out,
            "claam": claam_out,
            "multi_feats": multi_feats,
            "sdf_pred": sdf_pred,
        }

print('[cell3] DABLCNet architecture defined')


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 4 — DABLCLoss                                          ║
# ║                                                              ║
# ║  Asymmetric BCE  : w_in / w_out learned via log-ratio        ║
# ║    -> gradient flows through _logit_ratio (sum-constrained)   ║
# ║    -> NOT passed as weight= to F.bce (non-differentiable)     ║
# ║  Gap-aware boundary loss : penalises inter-building merging   ║
# ║  EDL loss        : Bayes risk + annealed KL (Contribution 1) ║
# ║  CLAAM loss      : CAM-BCE + contrastive + entropy floor      ║
# ║  CLAC loss       : cross-layer activation consistency         ║
# ║  SDR loss        : signed-distance regression                 ║
# ╚══════════════════════════════════════════════════════════════╝

class DABLCLoss(nn.Module):
    """DABL-C loss + Novel contributions:
    - Original: asymmetric BCE + Dice + Boundary sharpening + Confidence supervision
    - NEW: Constrained learnable weights (ratio-based, prevents collapse)
    - NEW: Gap-aware boundary loss (penalizes inter-building merging)
    - NEW: EDL Bayes risk + KL (Contribution 1)
    - NEW: CLAAM consistency + diversity regularization (Contribution 2 / MLAA)
    - NEW: CLAC spatial consistency regularization (Idea 2)
    """
    def __init__(self, w_in_init=1.0, w_out_init=1.0, dice_weight=1.0,
                 boundary_weight=0.5, conf_loss_weight=0.3):
        super().__init__()
        # Constrained parameterization: learn the LOG-RATIO of w_in/w_out
        # Total sum is fixed at (w_in_init + w_out_init), only the balance is learned.
        # This prevents both weights from collapsing to zero.
        self._total_w = w_in_init + w_out_init  # fixed constant
        # logit_ratio: sigmoid(logit) = w_in / (w_in + w_out)
        init_ratio = w_in_init / (w_in_init + w_out_init)
        init_logit = math.log(init_ratio / (1.0 - init_ratio + 1e-8) + 1e-8)
        self._logit_ratio = nn.Parameter(torch.tensor(float(init_logit)))
        self.dice_weight = dice_weight
        self.boundary_weight = boundary_weight
        self.conf_loss_weight = conf_loss_weight

    @property
    def w_in(self):
        """Building weight: derived from learned ratio, sum-constrained."""
        ratio = torch.sigmoid(self._logit_ratio)
        return self._total_w * ratio

    @property
    def w_out(self):
        """Background weight: derived from learned ratio, sum-constrained."""
        ratio = torch.sigmoid(self._logit_ratio)
        return self._total_w * (1.0 - ratio)

    def _boundary(self, x):
        """Gradient-based boundary extraction."""
        gx = x[:, :, :, 1:] - x[:, :, :, :-1]
        gy = x[:, :, 1:, :] - x[:, :, :-1, :]
        gx = F.pad(gx, (0, 1, 0, 0))
        gy = F.pad(gy, (0, 0, 0, 1))
        return (gx.abs() + gy.abs()).clamp(0, 1)

    def _gap_weight_map(self, target):
        """Create weight map emphasizing inter-building gaps.
        Uses GPU-based max-pool dilation to find gap pixels between
        adjacent buildings — these are the regions where merging errors
        occur and boundary accuracy is most critical.
        Returns: weight map [B, 1, H, W] with higher values at gaps."""
        # Dilate GT buildings on GPU using max pooling (equivalent to morphological dilation)
        dilated = F.max_pool2d(target, kernel_size=11, stride=1, padding=5)
        # Gap pixels: near buildings but not buildings themselves
        gap = (dilated - target).clamp(0, 1)
        # Building boundary pixels: buildings minus eroded buildings
        eroded = -F.max_pool2d(-target, kernel_size=5, stride=1, padding=2)  # min-pool = erosion
        boundary = (target - eroded).clamp(0, 1)
        # Weight map: base=1, gap regions=5x, building boundaries=3x
        weight = 1.0 + 4.0 * gap + 2.0 * boundary
        return weight

    def _claam_diversity_loss(self, layer_logits):
        """Penalize excessive agreement among CLAAM per-layer predictions.
        Encourages each ViT layer head to capture different building aspects.
        Uses pairwise cosine similarity on downsampled predictions."""
        if len(layer_logits) < 2:
            return torch.tensor(0.0, device=layer_logits[0].device)
        flat = [F.adaptive_avg_pool2d(torch.sigmoid(l), (16, 16)).reshape(l.shape[0], -1)
                for l in layer_logits]
        sim_sum = 0.0
        count = 0
        for i in range(len(flat)):
            for j in range(i + 1, len(flat)):
                sim_sum = sim_sum + F.cosine_similarity(flat[i], flat[j], dim=-1).mean()
                count += 1
        return sim_sum / max(count, 1)

    def forward(self, logits, target, conf=None, model_output=None,
                epoch=0, total_epochs=45, is_pseudo=False):
        probs = torch.sigmoid(logits)
        eps = 1e-6
        # NEW: track per-loss components for paper logging
        self.last_components = {}
        w_in = self.w_in     # derived from constrained ratio
        w_out = self.w_out

        # Dynamic pos_weight: small buildings get drowned by background
        pos_pixels = target.sum() + eps
        neg_pixels = (1 - target).sum() + eps
        pos_weight = (neg_pixels / pos_pixels).clamp(max=5.0)

        # Asymmetric BCE with area-based reweighting
        bce = -(w_in * pos_weight * target * torch.log(probs + eps) +
                w_out * (1 - target) * torch.log(1 - probs + eps))
        if conf is not None:
            bce = bce * conf
        bce_loss = bce.mean()
        self.last_components["bce"] = bce_loss.detach().item()

        # Dice loss
        inter = (probs * target).sum(dim=(2, 3))
        union = probs.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
        dice_loss = (1.0 - (2.0 * inter + eps) / (union + eps)).mean()
        self.last_components["dice"] = dice_loss.detach().item()

        # Gap-aware boundary loss: steep sigmoid -> near-hard edges
        # Weight boundary errors MORE heavily in inter-building gap regions
        # This directly penalizes the under-segmentation / merging problem
        sharp_probs = torch.sigmoid(logits * 10.0)
        pred_bd = self._boundary(sharp_probs)
        gt_bd = self._boundary(target)
        gap_weights = self._gap_weight_map(target)
        bd_loss = (gap_weights * (pred_bd - gt_bd) ** 2).mean()
        self.last_components["boundary"] = bd_loss.detach().item()

        total = bce_loss + self.dice_weight * dice_loss + self.boundary_weight * bd_loss

        # Scale factor for novel loss terms during pseudo-label training
        novel_scale = 0.3 if is_pseudo else 1.0

        # ── CONTRIBUTION 1: EDL Loss (Bayes risk + annealed KL) ──
        if model_output is not None and model_output.get("edl") is not None:
            alpha = model_output["edl"]["alpha"]
            l_edl = edl_loss(alpha, target, epoch, total_epochs,
                             annealing_epochs=cfg.edl_annealing_epochs)
            total = total + cfg.edl_loss_weight * novel_scale * l_edl
            self.last_components["edl"] = l_edl.detach().item()

        # ── CONTRIBUTION 2: CAM-Guided CLAAM v2 (contrastive + entropy floor) ──
        if model_output is not None and model_output.get("claam") is not None:
            claam_out = model_output["claam"]
            cam_logits = claam_out["cam_logits"]
            agreement = claam_out["agreement"]
            if cam_logits.shape[-2:] != target.shape[-2:]:
                cam_logits = F.interpolate(cam_logits, size=target.shape[-2:],
                                           mode="bilinear", align_corners=False)
            if agreement.shape[-2:] != target.shape[-2:]:
                agreement = F.interpolate(agreement, size=target.shape[-2:],
                                          mode="bilinear", align_corners=False)
            target_f = target.float()

            # (a) CAM supervision (BCE)
            cam_loss = F.binary_cross_entropy_with_logits(cam_logits.float(), target_f)

            # (b) Margin-based contrastive: agreement HIGH on FG, LOW on BG.
            # This is the anti-collapse term — without contrast, the agreement
            # map saturates to a constant.
            fg_pixels = target_f.sum().clamp_min(1.0)
            bg_pixels = (1.0 - target_f).sum().clamp_min(1.0)
            high = getattr(cfg, "claam_high_margin", 0.7)
            low  = getattr(cfg, "claam_low_margin",  0.3)
            fg_violation = F.relu(high - agreement) * target_f
            bg_violation = F.relu(agreement - low) * (1.0 - target_f)
            contrast_loss = (fg_violation.sum() / fg_pixels +
                             bg_violation.sum() / bg_pixels) * 0.5

            # (c) Entropy floor: penalize spatially uniform agreement maps
            sp_var = agreement.flatten(2).var(dim=2).mean()
            entropy_loss = F.relu(0.01 - sp_var) / 0.01

            l_claam = (cfg.claam_cam_weight * cam_loss
                       + cfg.claam_contrast_weight * contrast_loss
                       + cfg.claam_entropy_weight * entropy_loss)
            total = total + cfg.claam_loss_weight * novel_scale * l_claam
            self.last_components["claam_cam"] = cam_loss.detach().item()
            self.last_components["claam_contrast"] = contrast_loss.detach().item()
            self.last_components["claam_entropy"] = entropy_loss.detach().item()

        # ── IDEA 2: CLAC-Loss (cross-layer spatial consistency regularization) ──
        if model_output is not None and model_output.get("multi_feats") is not None:
            l_clac = clac_loss(model_output["multi_feats"])
            total = total + cfg.clac_loss_weight * l_clac
            self.last_components["clac"] = l_clac.detach().item()

        # ── IDEA 5: SDR Loss (signed distance field regression) ──
        if model_output is not None and model_output.get("sdf_pred") is not None:
            # FIX 10: skip SDR loss when pseudo-label has insufficient FG (junk SDF).
            fg_pixels_count = float(target.sum().item())
            if (not is_pseudo) or fg_pixels_count >= 256:
                sdf_pred = model_output["sdf_pred"]
                sdf_target = compute_sdf_target(target, max_dist=cfg.sdr_max_dist)
                l_sdr = F.smooth_l1_loss(sdf_pred, sdf_target)
                total = total + cfg.sdr_loss_weight * novel_scale * l_sdr
                self.last_components["sdr"] = l_sdr.detach().item()

        return total

print('[cell4] DABLCLoss defined')


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 5 — 7-gate pseudo-label filter + TVR buffer            ║
# ║                                                              ║
# ║  Gate 1 : Probability threshold (annealed)                   ║
# ║  Gate 2 : EDL uncertainty  (low uncertainty = trustworthy)   ║
# ║  Gate 3 : Edge proximity   (high edge = boundary pixel)      ║
# ║  Gate 4 : Geometry         (compactness / rectangularity)    ║
# ║  Gate 5 : CLAAM agreement  (cross-layer consensus)           ║
# ║  Gate 6 : GOP              (rectilinear boundary prior)      ║
# ║  Gate 7 : CAFCG            (frequency consistency)           ║
# ║                                                              ║
# ║  All 7 gates evaluated at COMPONENT level (not pixel level)  ║
# ║  TVR rolling window (bit-packed) also feeds into Gate 5      ║
# ╚══════════════════════════════════════════════════════════════╝


def progressive_threshold(epoch: int, total_epochs: int, start: float, end: float) -> float:
    if total_epochs <= 1:
        return end
    t = epoch / (total_epochs - 1)
    return start + t * (end - start)


def _geometry_gate(probs: torch.Tensor, thr: float = 0.5,
                   compact_min: float = 0.20, rect_min: float = 0.50,
                   max_aspect: float = 6.0, min_area: int = 20) -> torch.Tensor:
    """Geom v2: stricter rules - rejects tiny / non-rectangular / overly elongated comps."""
    import cv2
    B = probs.shape[0]
    gate = torch.ones_like(probs, dtype=torch.bool)
    for b in range(B):
        mask_np = (probs[b, 0].detach().cpu().numpy() > thr).astype(np.uint8) * 255
        if mask_np.sum() == 0:
            continue
        contours, _ = cv2.findContours(mask_np, cv2.RETR_EXTERNAL,
                                        cv2.CHAIN_APPROX_SIMPLE)
        reject_mask = np.zeros_like(mask_np, dtype=np.uint8)
        for cnt in contours:
            area = cv2.contourArea(cnt)
            if area < min_area:
                cv2.drawContours(reject_mask, [cnt], -1, 255, thickness=cv2.FILLED)
                continue
            peri = cv2.arcLength(cnt, True)
            compactness = (4 * math.pi * area) / (peri * peri + 1e-6)
            x, y, w, h = cv2.boundingRect(cnt)
            rectangularity = area / (w * h + 1e-6)
            aspect = max(w, h) / max(min(w, h), 1)
            if (compactness < compact_min or rectangularity < rect_min
                    or aspect > max_aspect):
                cv2.drawContours(reject_mask, [cnt], -1, 255, thickness=cv2.FILLED)
        if reject_mask.any():
            reject_t = torch.from_numpy(reject_mask > 0).to(probs.device)
            gate[b, 0] = gate[b, 0] & (~reject_t)
    return gate


def _erode_pseudo_labels(pseudo: torch.Tensor, erode_px: int) -> torch.Tensor:
    """Erode pseudo-label boundaries by erode_px pixels using GPU min-pooling.
    This PREVENTS the merging echo-chamber: by shrinking each predicted building
    a few pixels inward, we ensure the model never reinforces the 'bleed' across
    narrow inter-building gaps from its own noisy predictions.
    Only the confident interior pixels survive as pseudo-labels."""
    if erode_px <= 0:
        return pseudo
    k = 2 * erode_px + 1
    # Min-pooling = erosion for binary masks: shrinks foreground by erode_px
    eroded = -F.max_pool2d(-pseudo, kernel_size=k, stride=1, padding=erode_px)
    return eroded


# ═══════════════════════════════════════════════════════════════════
#  IDEA 3: Temporal Voting across Rotations (TVR)
# ═══════════════════════════════════════════════════════════════════

class TemporalVotingBuffer:
    """Rolling TVR history stored as binary masks.

    The first version kept full float32 masks in memory and on disk, which can
    make best_model_tvr.pt many GB. This version stores uint8 masks in RAM and
    bit-packs them for checkpoints while keeping the same voting behavior.
    """
    def __init__(self, window: int = 5, accept_ratio: float = 0.6):
        self.window = window
        self.accept_ratio = accept_ratio
        self.buffer = {}

    @staticmethod
    def _normalize_key(path_or_name):
        base = os.path.basename(str(path_or_name))
        return os.path.splitext(base)[0]

    def update(self, image_ids: list, predictions: torch.Tensor):
        from collections import deque
        preds = (predictions.detach().cpu() > 0.5).to(torch.uint8)
        for i, img_id in enumerate(image_ids):
            key = self._normalize_key(img_id)
            if key not in self.buffer:
                self.buffer[key] = deque(maxlen=self.window)
            self.buffer[key].append(preds[i:i+1].contiguous())

    def get_stable_mask(self, image_ids: list, device: torch.device) -> torch.Tensor:
        masks = []
        for img_id in image_ids:
            key = self._normalize_key(img_id)
            if key not in self.buffer or len(self.buffer[key]) < 2:
                masks.append(torch.ones(1, 1, 1, 1))
            else:
                stacked = torch.cat([m.float() for m in self.buffer[key]], dim=0)
                vote_frac = stacked.mean(dim=0, keepdim=True)
                stable = (vote_frac >= self.accept_ratio).float()
                masks.append(stable)
        return [m.to(device) for m in masks]

    @staticmethod
    def _pack_mask(mask: torch.Tensor) -> dict:
        arr = (mask.detach().cpu().numpy() > 0).astype(np.uint8)
        shape = tuple(arr.shape)
        packed = np.packbits(arr.reshape(-1))
        return {"shape": shape, "packed": torch.from_numpy(packed.copy())}

    @staticmethod
    def _unpack_mask(item) -> torch.Tensor:
        if isinstance(item, dict) and "packed" in item:
            shape = tuple(item["shape"])
            packed = item["packed"].detach().cpu().numpy().astype(np.uint8)
            flat = np.unpackbits(packed, count=int(np.prod(shape))).astype(np.uint8)
            return torch.from_numpy(flat.reshape(shape)).to(torch.uint8)
        if isinstance(item, torch.Tensor):
            return (item.detach().cpu() > 0.5).to(torch.uint8)
        return (torch.tensor(item) > 0.5).to(torch.uint8)

    def state_dict(self):
        return {
            "format": "tvr_packbits_v1",
            "window": self.window,
            "accept_ratio": self.accept_ratio,
            "buffer": {
                img_id: [self._pack_mask(pred) for pred in preds]
                for img_id, preds in self.buffer.items()
            },
        }

    def load_state_dict(self, state):
        from collections import deque
        self.window = int(state.get("window", self.window))
        self.accept_ratio = float(state.get("accept_ratio", self.accept_ratio))
        self.buffer = {}
        for img_id, preds in state.get("buffer", {}).items():
            q = deque(maxlen=self.window)
            for pred in preds:
                q.append(self._unpack_mask(pred).contiguous())
            self.buffer[self._normalize_key(img_id)] = q
        return self


def get_tvr_save_path(model_path=None):
    model_path = model_path or cfg.save_path
    root, ext = os.path.splitext(model_path)
    ext = ext or ".pt"
    return f"{root}_tvr{ext}"


def save_tvr_buffer(buffer=None, path=None):
    buffer = buffer or tvr_buffer
    path = path or get_tvr_save_path()
    torch.save(buffer.state_dict(), path)
    size_mb = os.path.getsize(path) / (1024 ** 2) if os.path.exists(path) else 0.0
    print(f"  TVR compact save: {len(buffer.buffer)} ids, {size_mb:.1f} MB -> {path}")
    return path


def load_tvr_buffer(path=None, buffer=None):
    buffer = buffer or tvr_buffer
    path = path or get_tvr_save_path()
    if not os.path.exists(path):
        print(f"No TVR buffer found at {path}")
        return False
    state = torch.load(path, map_location="cpu")
    buffer.load_state_dict(state)
    print(f"Loaded TVR buffer from {path} ({len(buffer.buffer)} image ids)")
    return True


tvr_buffer = TemporalVotingBuffer(
    window=cfg.tvr_window,
    accept_ratio=cfg.tvr_accept_ratio
)


# ═══════════════════════════════════════════════════════════════════
#  Legacy triple_gate_accept (kept for backward compat / ablation)
# ═══════════════════════════════════════════════════════════════════

def triple_gate_accept(probs: torch.Tensor, conf: torch.Tensor,
                       edge: torch.Tensor, thr: float) -> torch.Tensor:
    gate_conf = conf >= thr
    edge_strength = torch.sigmoid(edge)
    gate_edge = edge_strength <= 0.6
    gate_geom = _geometry_gate(probs, thr=0.5)
    gate = gate_conf & gate_edge & gate_geom
    b = probs.shape[0]
    pseudo = torch.zeros_like(probs)
    for i in range(b):
        p = probs[i]
        p_mean = p.mean()
        p_std = p.std()
        adaptive_thr = max(p_mean + 0.5 * p_std, 0.2)
        pseudo[i] = (p >= adaptive_thr).float()
    pseudo = pseudo * gate.float()
    return pseudo


# ═══════════════════════════════════════════════════════════════════
#  NEW: Multi-Uncertainty Gate (7 gates — all contributions combined)
# ═══════════════════════════════════════════════════════════════════

def _component_level_all7_from_gates(probs: torch.Tensor,
                                     edl_unc: torch.Tensor,
                                     edge_strength: torch.Tensor,
                                     claam_agree: torch.Tensor,
                                     tvr_map: torch.Tensor,
                                     gop_gate: torch.Tensor,
                                     cafcg_gate_map: torch.Tensor,
                                     prob_thresh: float = 0.5,
                                     min_area: int = 16) -> torch.Tensor:
    """Component-level all-7 pseudo-label filter.

    Raw predicted connected components are accepted/rejected as objects. Each
    gate contributes a component-level score, avoiding pixel-wise anti-correlation
    between interior gates (EDL/Edge) and boundary gates (GOP/CAFCG).
    """
    from scipy import ndimage

    quorums = {
        "prob": 0.68,   # Component mean probability confidence
        "edl": 0.65,    # Require most component pixels to be low-uncertainty
        "edge": 0.50,   # Adaptive edge threshold handles scale
        "geom": 0.18,   # Remove thin line artifacts
        "claam": 0.01,  # Harmless until CLAAM is retrained out of collapse
        "tvr": 0.50,
        "gop": 0.30,    # Require stronger rectilinear evidence at component level
        "cafcg": 0.40,  # CAFCG signal is weak, so keep this slightly loose
    }

    final = torch.zeros_like(probs)
    for b in range(probs.shape[0]):
        prob_np = probs[b, 0].detach().cpu().float().numpy()
        raw_mask = prob_np >= prob_thresh
        labeled, n_comps = ndimage.label(raw_mask)
        if n_comps == 0:
            continue

        edl_np = edl_unc[b, 0].detach().cpu().float().numpy()
        edge_np = edge_strength[b, 0].detach().cpu().float().numpy()
        claam_np = claam_agree[b, 0].detach().cpu().float().numpy()
        tvr_np = tvr_map[b, 0].detach().cpu().float().numpy()
        gop_np = gop_gate[b, 0].detach().cpu().float().numpy()
        cafcg_np = cafcg_gate_map[b, 0].detach().cpu().float().numpy()

        edge_thresh = np.percentile(edge_np[raw_mask], 85) if raw_mask.any() else 0.7

        edl_pass = edl_np <= getattr(cfg, "edl_thr", 0.60)
        edge_pass = edge_np <= edge_thresh
        claam_pass = claam_np >= getattr(cfg, "claam_thr_end", 0.40)
        tvr_pass = tvr_np >= getattr(cfg, "tvr_thr", 0.50)
        gop_pass = gop_np >= getattr(cfg, "gop_thr", 0.25)
        # cafcg_gate_map is a pass mask here (1=pass, 0=reject), not raw disagreement.
        cafcg_pass = cafcg_np >= 0.5

        final_np = np.zeros_like(raw_mask, dtype=np.float32)
        for comp_id in range(1, n_comps + 1):
            comp = labeled == comp_id
            area = int(comp.sum())
            if area < min_area:
                continue

            rows, cols = np.where(comp)
            bbox_h = int(rows.max() - rows.min()) + 1
            bbox_w = int(cols.max() - cols.min()) + 1
            compactness = area / max(float(bbox_h * bbox_w), 1.0)

            prob_score = float(prob_np[comp].mean())
            passes = (
                prob_score >= quorums["prob"] and
                edl_pass[comp].mean() >= quorums["edl"] and
                edge_pass[comp].mean() >= quorums["edge"] and
                compactness >= quorums["geom"] and
                claam_pass[comp].mean() >= quorums["claam"] and
                tvr_pass[comp].mean() >= quorums["tvr"] and
                gop_pass[comp].mean() >= quorums["gop"] and
                cafcg_pass[comp].mean() >= quorums["cafcg"]
            )
            if passes:
                # Component-level decision, high-precision pixel emission.
                final_np[comp & edl_pass] = 1.0

        final[b, 0] = torch.from_numpy(final_np).to(probs.device, dtype=probs.dtype)
    return final


def multi_uncertainty_gate(probs: torch.Tensor, model_output: dict,
                           thr: float, epoch: int, total_epochs: int,
                           image_ids: list = None) -> torch.Tensor:
    """7-gate component-level pseudo-label acceptance.

    All seven gates are retained, but they are evaluated over each predicted
    building component instead of intersected pixel-by-pixel.
    """
    # AMP fix: under autocast, probs/edge/edl/claam are float16. Downstream gate
    # ops (cv2.Sobel, F.conv2d with float32 kernels) require float32. Cast once
    # at the entry point so every gate is safe.
    probs = probs.float()
    edge = model_output.get("edge")
    if edge is not None:
        edge = edge.float()
    edl_out = model_output.get("edl")
    if edl_out is not None and "uncertainty" in edl_out:
        edl_out = {**edl_out, "uncertainty": edl_out["uncertainty"].float()}
    claam_out = model_output.get("claam")
    if claam_out is not None and "agreement" in claam_out:
        claam_out = {**claam_out, "agreement": claam_out["agreement"].float()}
    B = probs.shape[0]

    edl_unc = edl_out["uncertainty"]
    edge_strength = torch.sigmoid(edge)
    claam_agreement = claam_out["agreement"]

    if image_ids is not None:
        tvr_masks = tvr_buffer.get_stable_mask(image_ids, probs.device)
        gate_tvr_list = []
        for i in range(B):
            m = tvr_masks[i] if i < len(tvr_masks) else torch.ones(1, 1, 1, 1, device=probs.device)
            if m.shape[-2:] != probs.shape[-2:]:
                m = torch.ones(1, 1, probs.shape[2], probs.shape[3], device=probs.device)
            gate_tvr_list.append(m.float())
        tvr_map = torch.cat(gate_tvr_list, dim=0)
    else:
        tvr_map = torch.ones_like(probs)

    # Existing GOP/CAFCG implementations return pass/fail masks; component-level
    # quorum turns those pixel masks into object-level scores.
    gop_gate = gradient_orientation_gate(probs, thr=0.5, rectilinear_thr=0.25).float()
    cafcg_gate_map = cafcg_gate(probs, disagreement_thr=max(cfg.cafcg_disagreement_thr, 0.40)).float()

    pseudo = _component_level_all7_from_gates(
        probs=probs,
        edl_unc=edl_unc,
        edge_strength=edge_strength,
        claam_agree=claam_agreement,
        tvr_map=tvr_map,
        gop_gate=gop_gate,
        cafcg_gate_map=cafcg_gate_map,
        prob_thresh=0.5,
        min_area=16,
    )

    if image_ids is not None:
        tvr_buffer.update(image_ids, pseudo)

    return pseudo

print('[cell5] 7-gate filter + TVR buffer defined')


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 6 — Evaluation metrics                                 ║
# ║                                                              ║
# ║  boundary_iou   BIoU — PRIMARY metric (best-model trigger)   ║
# ║  standard_iou   pixel-level IoU / Jaccard                    ║
# ║  hausdorff_distance  HD95 (boundary accuracy)                ║
# ║  assd_distance  Average Symmetric Surface Distance           ║
# ║  ece_score      Expected Calibration Error                   ║
# ╚══════════════════════════════════════════════════════════════╝

from scipy.ndimage import binary_dilation, binary_erosion, distance_transform_edt


def boundary_iou(pred: torch.Tensor, target: torch.Tensor,
                 thr: float = 0.5, dilation_px: int = 3, eps: float = 1e-6) -> torch.Tensor:
    """Boundary-IoU following Cheng et al. 2021.
    Uses morphological dilation to extract boundary regions at fixed pixel tolerance,
    then computes IoU only within those boundary strips."""
    pred_bin = (pred >= thr).float()
    b = pred_bin.shape[0]
    struct = np.ones((dilation_px * 2 + 1, dilation_px * 2 + 1))
    biou_vals = []
    for i in range(b):
        p = pred_bin[i, 0].cpu().numpy()  # [H, W]
        t = target[i, 0].cpu().numpy()
        # Boundary = dilated XOR original (the boundary strip)
        p_dilated = binary_dilation(p, structure=struct).astype(np.float32)
        t_dilated = binary_dilation(t, structure=struct).astype(np.float32)
        p_eroded = binary_erosion(p, structure=struct).astype(np.float32)
        t_eroded = binary_erosion(t, structure=struct).astype(np.float32)
        p_boundary = np.clip((p_dilated - p) + (p - p_eroded), 0, 1)
        t_boundary = np.clip((t_dilated - t) + (t - t_eroded), 0, 1)
        # IoU within boundary regions
        inter = (p_boundary * t_boundary).sum()
        union = p_boundary.sum() + t_boundary.sum() - inter
        biou_vals.append((inter + eps) / (union + eps))
    return torch.tensor(np.mean(biou_vals), device=pred.device)


def standard_iou(pred: torch.Tensor, target: torch.Tensor,
                 thr: float = 0.5, eps: float = 1e-6) -> torch.Tensor:
    """Standard pixel-level IoU (Jaccard index)."""
    pred_bin = (pred >= thr).float()
    inter = (pred_bin * target).sum(dim=(1, 2, 3))
    union = pred_bin.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3)) - inter
    return ((inter + eps) / (union + eps)).mean()


def hausdorff_distance(pred: torch.Tensor, target: torch.Tensor,
                       thr: float = 0.5, percentile: float = 95.0) -> torch.Tensor:
    """Hausdorff distance via boundary distance transforms (robust: uses percentile).
    Uses boundary pixels (not filled regions) for standard HD95 computation.
    Properly handles empty pred/GT cases."""
    pred_bin = (pred >= thr).float()
    b = pred_bin.shape[0]
    hd_vals = []
    for i in range(b):
        p = pred_bin[i, 0].cpu().numpy().astype(bool)
        t = target[i, 0].cpu().numpy().astype(bool)
        # Both empty -> perfect agreement -> HD = 0
        if not p.any() and not t.any():
            hd_vals.append(0.0)
            continue
        # One empty, other not -> max possible distance as penalty (capped)
        if not p.any() or not t.any():
            hd_vals.append(float(np.sqrt(p.shape[0]**2 + p.shape[1]**2)))
            continue
        # Extract boundaries for standard HD95
        p_boundary = p ^ binary_erosion(p)
        t_boundary = t ^ binary_erosion(t)
        if not p_boundary.any():
            p_boundary = p  # single-pixel components
        if not t_boundary.any():
            t_boundary = t
        dt_pred = distance_transform_edt(~p_boundary)
        dt_tgt = distance_transform_edt(~t_boundary)
        d_t2p = dt_pred[t_boundary]   # distances from GT boundary to nearest pred boundary
        d_p2t = dt_tgt[p_boundary]    # distances from pred boundary to nearest GT boundary
        hd = max(np.percentile(d_t2p, percentile), np.percentile(d_p2t, percentile))
        hd_vals.append(float(hd))
    return torch.tensor(np.mean(hd_vals), device=pred.device)


def assd_distance(pred: torch.Tensor, target: torch.Tensor,
                  thr: float = 0.5) -> torch.Tensor:
    """Average Symmetric Surface Distance via distance transforms.
    ASSD = mean of all surface-to-surface distances in both directions.
    Properly handles empty pred/GT cases."""
    pred_bin = (pred >= thr).float()
    b = pred_bin.shape[0]
    assd_vals = []
    for i in range(b):
        p = pred_bin[i, 0].cpu().numpy().astype(bool)
        t = target[i, 0].cpu().numpy().astype(bool)
        # Both empty -> perfect agreement -> ASSD = 0
        if not p.any() and not t.any():
            assd_vals.append(0.0)
            continue
        # One empty, other not -> max penalty
        if not p.any() or not t.any():
            assd_vals.append(float(np.sqrt(p.shape[0]**2 + p.shape[1]**2)))
            continue
        p_boundary = p ^ binary_erosion(p)
        t_boundary = t ^ binary_erosion(t)
        if not p_boundary.any():
            p_boundary = p
        if not t_boundary.any():
            t_boundary = t
        dt_pred = distance_transform_edt(~p_boundary)
        dt_tgt = distance_transform_edt(~t_boundary)
        d_t2p = dt_pred[t_boundary].mean() if t_boundary.any() else 0.0
        d_p2t = dt_tgt[p_boundary].mean() if p_boundary.any() else 0.0
        assd_vals.append(float((d_t2p + d_p2t) / 2.0))
    return torch.tensor(np.mean(assd_vals), device=pred.device)


def ece_score(probs: torch.Tensor, target: torch.Tensor, n_bins: int = 15) -> torch.Tensor:
    conf = probs.view(-1)
    t = target.view(-1)
    bins = torch.linspace(0, 1, n_bins + 1, device=probs.device)
    ece = torch.zeros(1, device=probs.device)
    for i in range(n_bins):
        in_bin = (conf >= bins[i]) & (conf < bins[i + 1])
        if in_bin.any():
            acc = t[in_bin].mean()
            avg_conf = conf[in_bin].mean()
            ece += (in_bin.float().mean()) * (acc - avg_conf).abs()
    return ece


def reliability_bins(probs: torch.Tensor, target: torch.Tensor, n_bins: int = 15):
    conf = probs.view(-1)
    t = target.view(-1)
    bins = torch.linspace(0, 1, n_bins + 1, device=probs.device)
    bin_acc, bin_conf, bin_frac = [], [], []
    for i in range(n_bins):
        in_bin = (conf >= bins[i]) & (conf < bins[i + 1])
        if in_bin.any():
            bin_acc.append(t[in_bin].mean().item())
            bin_conf.append(conf[in_bin].mean().item())
            bin_frac.append(in_bin.float().mean().item())
        else:
            bin_acc.append(0.0)
            bin_conf.append(0.0)
            bin_frac.append(0.0)
    return bin_acc, bin_conf, bin_frac


print('[cell6] metrics defined  (BIoU · IoU · HD95 · ASSD · ECE)')


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 7 — WHU Dataset + DataLoaders                          ║
# ║                                                              ║
# ║  WHUDataset        joint augment (flip/rotate/colour jitter) ║
# ║  fix_labeled_split seeded once at seed=42, never reshuffles  ║
# ║  get_train_loaders returns (labeled, unlabeled) each epoch   ║
# ║  build_dataloaders applies smoke caps when SMOKE_TEST=True   ║
# ╚══════════════════════════════════════════════════════════════╝

DATA_ROOT  = cfg.data_root
TRAIN_IMG  = os.path.join(DATA_ROOT, 'train', 'Image')
TRAIN_MASK = os.path.join(DATA_ROOT, 'train', 'Mask')
VAL_IMG    = os.path.join(DATA_ROOT, 'val',   'Image')
VAL_MASK   = os.path.join(DATA_ROOT, 'val',   'Mask')
TEST_IMG   = os.path.join(DATA_ROOT, 'test',  'Image')
TEST_MASK  = os.path.join(DATA_ROOT, 'test',  'Mask')

# Global train paths — populated by build_dataloaders()
ALL_TRAIN_IMG  = []
ALL_TRAIN_MASK = []

# Fixed split globals — populated by fix_labeled_split()
_LABELED_IMG   = []
_LABELED_MASK  = []
_UNLABELED_IMG = []

class RandomSegDataset(Dataset):
    def __init__(self, length: int = 10, img_size: int = 512):
        self.length = length
        self.img_size = img_size

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        img = torch.rand(3, self.img_size, self.img_size)
        mask = (torch.rand(1, self.img_size, self.img_size) > 0.5).float()
        return img, mask


def load_images(folder: str):
    exts = ["*.png", "*.jpg", "*.jpeg", "*.tif", "*.bmp"]
    paths = []
    for ext in exts:
        paths += glob.glob(os.path.join(folder, ext))
    return sorted(paths)




class WHUDataset(Dataset):
    """WHU Building dataset with optional joint augmentation."""
    def __init__(self, image_paths, mask_paths=None, img_size=512, augment=False, return_id=False):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.img_size = img_size
        self.augment = augment
        self.return_id = return_id

    def __len__(self):
        return len(self.image_paths)

    def _augment(self, img, mask=None):
        """Joint spatial + photometric augmentation for image (and mask)."""
        if random.random() > 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
            if mask is not None:
                mask = mask.transpose(Image.FLIP_LEFT_RIGHT)
        if random.random() > 0.5:
            img = img.transpose(Image.FLIP_TOP_BOTTOM)
            if mask is not None:
                mask = mask.transpose(Image.FLIP_TOP_BOTTOM)
        k = random.randint(0, 3)
        if k == 1:
            img = img.transpose(Image.ROTATE_90)
            if mask is not None: mask = mask.transpose(Image.ROTATE_90)
        elif k == 2:
            img = img.transpose(Image.ROTATE_180)
            if mask is not None: mask = mask.transpose(Image.ROTATE_180)
        elif k == 3:
            img = img.transpose(Image.ROTATE_270)
            if mask is not None: mask = mask.transpose(Image.ROTATE_270)

        img_np = np.array(img).astype(np.float32)
        img_np = img_np * random.uniform(0.8, 1.2)
        mean_val = img_np.mean()
        img_np = (img_np - mean_val) * random.uniform(0.8, 1.2) + mean_val
        gray = img_np.mean(axis=2, keepdims=True)
        sat = random.uniform(0.7, 1.3)
        img_np = img_np * sat + gray * (1 - sat)
        img_np = np.clip(img_np, 0, 255).astype(np.uint8)
        img = Image.fromarray(img_np)
        return img, mask

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        img = Image.open(image_path).convert("RGB")
        img = img.resize((self.img_size, self.img_size), resample=Image.BILINEAR)

        mask = None
        if self.mask_paths:
            mask = Image.open(self.mask_paths[idx]).convert("L")
            mask = mask.resize((self.img_size, self.img_size), resample=Image.NEAREST)

        if self.augment:
            img, mask = self._augment(img, mask)

        img = torch.tensor(np.array(img) / 255.0).permute(2, 0, 1).float()

        if mask is not None:
            mask = (np.array(mask) > 127).astype(np.float32)
            mask = torch.tensor(mask).unsqueeze(0)
            if self.return_id:
                return img, mask, os.path.basename(image_path)
            return img, mask

        if self.return_id:
            return img, os.path.basename(image_path)

        return img


# ── Store ALL train paths globally for label rotation ──
ALL_TRAIN_IMG = []
ALL_TRAIN_MASK = []

# FIX 11: FIXED labeled subset — seeded once, never reshuffles.
# This is what the paper means when it claims "10% labels semi-supervised".
# The OLD rotate_train_loaders() reshuffled every call, so over 30+ epochs
# the model effectively saw ~96% of images with labels.

_LABELED_IMG = []
_LABELED_MASK = []
_UNLABELED_IMG = []


def fix_labeled_split(label_frac=None, seed=42):
    global _LABELED_IMG, _LABELED_MASK, _UNLABELED_IMG
    if label_frac is None:
        label_frac = cfg.label_frac
    n = len(ALL_TRAIN_IMG)
    if n == 0:
        return
    n_labeled = max(1, int(n * label_frac))
    rng = random.Random(seed)
    indices = list(range(n))
    rng.shuffle(indices)
    lab_idx = indices[:n_labeled]
    unl_idx = indices[n_labeled:]
    _LABELED_IMG  = [ALL_TRAIN_IMG[i]  for i in lab_idx]
    _LABELED_MASK = [ALL_TRAIN_MASK[i] for i in lab_idx]
    _UNLABELED_IMG = [ALL_TRAIN_IMG[i] for i in unl_idx]
    print(f"[fix_labeled_split] seed={seed} | labeled={len(_LABELED_IMG)} "
          f"| unlabeled={len(_UNLABELED_IMG)} "
          f"| labeled fraction = {len(_LABELED_IMG)/n:.2%}")


def get_train_loaders():
    """Return loaders backed by the FIXED split.

    Both loaders set return_id=True so the training loop receives stable
    filename IDs (FIX 3 — TVR uses these instead of pixel hashes).
    """
    if not _LABELED_IMG:
        raise RuntimeError("Call fix_labeled_split() first.")
    lab_ds = WHUDataset(_LABELED_IMG, _LABELED_MASK,
                        img_size=cfg.img_size, augment=True, return_id=True)
    unl_ds = WHUDataset(_UNLABELED_IMG, None,
                        img_size=cfg.img_size, augment=True, return_id=True)
    lab_loader = DataLoader(lab_ds, batch_size=cfg.batch_size,
                            shuffle=True, drop_last=True,
                            num_workers=cfg.num_workers, pin_memory=True,
                            persistent_workers=(cfg.num_workers > 0))
    unl_loader = DataLoader(unl_ds, batch_size=cfg.batch_size,
                            shuffle=True, drop_last=True,
                            num_workers=cfg.num_workers, pin_memory=True,
                            persistent_workers=(cfg.num_workers > 0))
    return lab_loader, unl_loader


# Legacy alias for any external code; uses the fixed split now.
def rotate_train_loaders():
    return get_train_loaders()


def build_dataloaders():
    global ALL_TRAIN_IMG, ALL_TRAIN_MASK
    if os.path.isdir(DATA_ROOT):
        train_img = load_images(TRAIN_IMG)
        train_mask = load_images(TRAIN_MASK)
        val_img = load_images(VAL_IMG)
        val_mask = load_images(VAL_MASK)

        n_train = min(len(train_img), len(train_mask))
        ALL_TRAIN_IMG = train_img[:n_train]
        ALL_TRAIN_MASK = train_mask[:n_train]
        # ── smoke-test hard cap: tiny dataset for 5-min runs ──
        if SMOKE_TEST:
            ALL_TRAIN_IMG  = ALL_TRAIN_IMG[:cfg.smoke_max_train]
            ALL_TRAIN_MASK = ALL_TRAIN_MASK[:cfg.smoke_max_train]
            val_img  = val_img[:cfg.smoke_max_val]
            val_mask = val_mask[:cfg.smoke_max_val]
            print(f'[smoke] train capped to {len(ALL_TRAIN_IMG)} '
                  f'| val capped to {len(val_img)}')

        n_val = min(len(val_img), len(val_mask))
        val_img = val_img[:n_val]
        val_mask = val_mask[:n_val]

        val_ds = WHUDataset(val_img, val_mask, img_size=cfg.img_size, augment=False)
    else:
        print("WHU paths not found, using random dataset.")
        val_img = []; val_mask = []
        val_ds = RandomSegDataset(length=2, img_size=cfg.img_size)

    # FIX 11: build labeled subset ONCE, seeded.
    if ALL_TRAIN_IMG:
        fix_labeled_split(label_frac=cfg.label_frac, seed=42)
    labeled_loader, unlabeled_loader = get_train_loaders() if ALL_TRAIN_IMG else (
        DataLoader(RandomSegDataset(length=4, img_size=cfg.img_size), batch_size=cfg.batch_size, shuffle=True),
        DataLoader(RandomSegDataset(length=2, img_size=cfg.img_size), batch_size=cfg.batch_size, shuffle=True),
    )
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False,
                            num_workers=cfg.num_workers, pin_memory=True,
                            persistent_workers=(cfg.num_workers > 0))
    n_labeled = max(1, int(len(ALL_TRAIN_IMG) * cfg.label_frac))
    print(f"[data] total train={len(ALL_TRAIN_IMG)}  "
          f"labeled={n_labeled}  "
          f"unlabeled={len(ALL_TRAIN_IMG)-n_labeled}  "
          f"val={len(val_img)}")
    return labeled_loader, unlabeled_loader, val_loader


# Build dataloaders (fix_labeled_split called inside)
labeled_loader, unlabeled_loader, val_loader = build_dataloaders()
print('[cell7] dataloaders ready')


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 8 — Checkpoint persistence                             ║
# ║                                                              ║
# ║  persist_files()           -> copies model+ckpt+TVR to        ║
# ║                              persist_dir every epoch         ║
# ║  try_restore_from_input()  -> on session resume, copies back  ║
# ║                              persisted files to /working     ║
# ║                                                              ║
# ║  HOW TO RESUME after session kill:                           ║
# ║    1. Save /kaggle/working/persist/ as Kaggle dataset output ║
# ║    2. Mount it as input on next session                      ║
# ║    3. Set RESUME_INPUT to the mounted path below             ║
# ║    4. Run all cells — training continues from last epoch     ║
# ╚══════════════════════════════════════════════════════════════╝

# Set this to your persisted dataset input path on resume, e.g.:
# RESUME_INPUT = "/kaggle/input/georect-whu-persist"
RESUME_INPUT = ""

def persist_files():
    """Copy latest model + checkpoint + TVR to persist_dir.
    Called every epoch so a session kill loses at most one epoch."""
    os.makedirs(cfg.persist_dir, exist_ok=True)
    pairs = [
        (cfg.save_path,       os.path.join(cfg.persist_dir, "best_model.pt")),
        (cfg.checkpoint_path, os.path.join(cfg.persist_dir, "training_checkpoint.pt")),
    ]
    tvr_src = get_tvr_save_path()
    if os.path.exists(tvr_src):
        pairs.append((tvr_src, os.path.join(cfg.persist_dir, "tvr_buffer.pt")))
    for src_path, dst in pairs:
        if os.path.exists(src_path):
            try:
                shutil.copy2(src_path, dst)
            except Exception as e:
                print(f"  [persist] {os.path.basename(src_path)}: {e}")


def try_restore_from_input():
    """On resume: copy persisted files from read-only input -> /kaggle/working."""
    if not RESUME_INPUT:
        return
    mapping = {
        "training_checkpoint.pt": cfg.checkpoint_path,
        "best_model.pt":          cfg.save_path,
        "tvr_buffer.pt":          get_tvr_save_path(),
    }
    restored = []
    for fname, dst in mapping.items():
        src_path = os.path.join(RESUME_INPUT, fname)
        if os.path.exists(src_path) and not os.path.exists(dst):
            shutil.copy2(src_path, dst)
            restored.append(fname)
    if restored:
        print(f"[restore] copied from {RESUME_INPUT}: {', '.join(restored)}")
    else:
        print(f"[restore] nothing to copy from {RESUME_INPUT}")


try_restore_from_input()
print(f"[cell8] persist_dir = {cfg.persist_dir}")
if RESUME_INPUT:
    print(f"[cell8] resuming from: {RESUME_INPUT}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 9 — Training loop                                      ║
# ║                                                              ║
# ║  Helpers  : AsyncGateCache · TrainingLogger · validate_full  ║
# ║  Stages   : Warmup (frozen backbone) -> S1 -> S2 -> S3          ║
# ║  Pseudo   : proportional budget (3× labeled) OR smoke cap    ║
# ║  Persist  : every epoch + immediately after new best BIoU    ║
# ║  Resume   : auto-detects checkpoint; skips completed stages  ║
# ║  AMP      : torch.amp.autocast/GradScaler (PyTorch ≥ 2.4)   ║
# ╚══════════════════════════════════════════════════════════════╝

import copy
import gc
import cv2
from scipy import ndimage as ndi
from scipy.ndimage import uniform_filter, label as cc_label
import hashlib

@torch.no_grad()
def check_claam_health(model, val_loader, device, n_batches=4):
    """Quick CLAAM-health diagnostic. Healthy means gap > 0.05 and sp_std > 0.05."""
    model.eval()
    bld, bg, sp_std = [], [], []
    seen = 0
    for batch in val_loader:
        if seen >= n_batches:
            break
        imgs, masks = batch[0], batch[1]
        imgs, masks = imgs.to(device), masks.to(device)
        out = model(imgs)
        agreement = out["claam"]["agreement"]
        gt = (masks > 0.5).float()
        bld.append((agreement * gt).sum().item() / max(gt.sum().item(), 1.0))
        bg.append((agreement * (1 - gt)).sum().item() / max((1 - gt).sum().item(), 1.0))
        sp_std.append(agreement.flatten(1).std(dim=1).mean().item())
        seen += 1
    bld_m = sum(bld) / max(len(bld), 1)
    bg_m = sum(bg) / max(len(bg), 1)
    return {"bld": bld_m, "bg": bg_m, "gap": bld_m - bg_m,
            "sp_std": sum(sp_std) / max(len(sp_std), 1)}



# ──────────────────────── Post-processing ────────────────────────

def morpho_clean(pred_np, ksize=3):
    """Morphological close only (fill holes). No opening — it destroys small buildings."""
    struct = np.ones((ksize, ksize))
    pred_np = ndi.binary_closing(pred_np, structure=struct).astype(np.float32)
    return pred_np


def watershed_separate(binary_np, min_distance=8, min_area=30):
    """Watershed-based building separation to split merged buildings.
    Uses distance transform + local maxima as markers."""
    from scipy.ndimage import label as nd_label, distance_transform_edt
    if binary_np.sum() < min_area:
        return binary_np
    dist = distance_transform_edt(binary_np)
    from scipy.ndimage import maximum_filter
    local_max = (dist == maximum_filter(dist, size=min_distance * 2 + 1))
    local_max = local_max & (dist > 2)
    markers, num_markers = nd_label(local_max)
    if num_markers < 2:
        return binary_np
    from skimage.segmentation import watershed
    labels = watershed(-dist, markers, mask=binary_np.astype(bool))
    result = np.zeros_like(binary_np, dtype=np.float32)
    for lbl in range(1, num_markers + 1):
        component = (labels == lbl).astype(np.float32)
        if component.sum() >= min_area:
            result = np.maximum(result, component)
    return result


def polygon_refine(binary_np, epsilon_frac=0.005, min_area=20):
    """Simplify each connected component's contour to a polygon."""
    mask = (binary_np * 255).astype(np.uint8)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    result = np.zeros_like(binary_np, dtype=np.float32)
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < min_area:
            continue
        peri = cv2.arcLength(cnt, True)
        if area < 200:
            epsilon = 0.002 * peri
        elif area < 800:
            epsilon = 0.004 * peri
        else:
            epsilon = epsilon_frac * peri
        approx = cv2.approxPolyDP(cnt, epsilon, True)
        cv2.drawContours(result, [approx], -1, 1.0, thickness=cv2.FILLED)
    return result


def refine_prediction(img_np, prob_np, thr=0.5, min_area=20):
    binary = (prob_np > thr).astype(np.float32)
    binary = morpho_clean(binary, ksize=3)
    binary = watershed_separate(binary, min_distance=8, min_area=min_area)
    sharp = polygon_refine(binary, epsilon_frac=0.005, min_area=min_area)
    return prob_np, sharp


def guided_filter(guide, src, radius=8, eps=0.01):
    g = guide.mean(axis=2).astype(np.float64) if guide.ndim == 3 else guide.astype(np.float64)
    s = src.astype(np.float64)
    sz = 2 * radius + 1
    mean_g  = uniform_filter(g, size=sz)
    mean_s  = uniform_filter(s, size=sz)
    mean_gs = uniform_filter(g * s, size=sz)
    mean_gg = uniform_filter(g * g, size=sz)
    a = (mean_gs - mean_g * mean_s) / (mean_gg - mean_g * mean_g + eps)
    b = mean_s - a * mean_g
    mean_a = uniform_filter(a, size=sz)
    mean_b = uniform_filter(b, size=sz)
    return np.clip(mean_a * g + mean_b, 0, 1).astype(np.float32)


def _make_image_ids(imgs: torch.Tensor) -> list:
    ids = []
    for i in range(imgs.shape[0]):
        small = F.interpolate(imgs[i:i+1], size=(32, 32), mode='bilinear', align_corners=False)
        h = hashlib.md5(small.cpu().numpy().tobytes()).hexdigest()[:12]
        ids.append(h)
    return ids



# ════════════════════════════════════════════════════════════════════
# Async pseudo-label cache (CPU/GPU parallelism)
# ════════════════════════════════════════════════════════════════════
import threading
import queue as queue_mod
import csv
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor


class AsyncGateCache:
    """Background CPU thread computes pseudo-labels while GPU trains.

    Bounded LRU cache to prevent memory blowup over long training runs.
    """
    def __init__(self, gate_fn, max_inflight=2, max_entries=1500):
        from collections import OrderedDict
        self.gate_fn = gate_fn
        self.executor = ThreadPoolExecutor(max_workers=1)
        self.cache = OrderedDict()   # LRU: image_id -> CPU pseudo tensor
        self.lock = threading.Lock()
        self.in_flight = set()
        self.max_inflight = max_inflight
        self.max_entries = max_entries
        self.pending = []
        self.stats = {"submitted": 0, "completed": 0, "hits": 0,
                      "misses": 0, "evicted": 0}

    def submit(self, probs_cpu, model_out_cpu, image_ids, epoch, total_epochs, thr):
        """Non-blocking. Submits gate computation to background thread."""
        # Skip ids already being computed
        new_idx = []
        new_ids = []
        with self.lock:
            for i, iid in enumerate(image_ids):
                if iid not in self.in_flight:
                    self.in_flight.add(iid)
                    new_idx.append(i)
                    new_ids.append(iid)
        if not new_ids:
            return
        # Slice tensors to only new ids
        probs_sub = probs_cpu[new_idx]
        out_sub = {
            "edge": model_out_cpu["edge"][new_idx],
            "edl": {"uncertainty": model_out_cpu["edl"]["uncertainty"][new_idx]},
            "claam": {"agreement": model_out_cpu["claam"]["agreement"][new_idx]},
        }
        self.executor.submit(self._compute, probs_sub, out_sub,
                              new_ids, epoch, total_epochs, thr)
        self.stats["submitted"] += len(new_ids)

    def _compute(self, probs, model_out, image_ids, epoch, total_epochs, thr):
        try:
            pseudo = self.gate_fn(probs, model_out, thr, epoch, total_epochs,
                                   image_ids=image_ids)
            with self.lock:
                for i, iid in enumerate(image_ids):
                    # LRU: pop existing, re-insert at end
                    if iid in self.cache:
                        del self.cache[iid]
                    self.cache[iid] = pseudo[i:i+1].detach().cpu()
                    self.in_flight.discard(iid)
                    self.stats["completed"] += 1
                # Evict oldest to keep cache bounded
                while len(self.cache) > self.max_entries:
                    self.cache.popitem(last=False)
                    self.stats["evicted"] += 1
        except Exception as e:
            with self.lock:
                for iid in image_ids:
                    self.in_flight.discard(iid)
            print(f"  [async gate error] {type(e).__name__}: {e}")

    def get(self, image_ids):
        """Returns stacked pseudo tensor on CPU, or None if any id is missing."""
        with self.lock:
            results = []
            for iid in image_ids:
                if iid in self.cache:
                    # LRU touch: move to end
                    self.cache.move_to_end(iid)
                    results.append(self.cache[iid])
                else:
                    self.stats["misses"] += 1
                    return None
            self.stats["hits"] += 1
        return torch.cat(results, dim=0)

    def shutdown(self):
        self.executor.shutdown(wait=True, cancel_futures=False)


def _model_output_to_cpu(out):
    """Move only the parts needed by gates to CPU. Robust to missing keys."""
    result = {}
    if out.get("edge") is not None:
        result["edge"] = out["edge"].detach().float().cpu()
    edl = out.get("edl") or {}
    if edl.get("uncertainty") is not None:
        result["edl"] = {"uncertainty": edl["uncertainty"].detach().float().cpu()}
    claam = out.get("claam") or {}
    if claam.get("agreement") is not None:
        result["claam"] = {"agreement": claam["agreement"].detach().float().cpu()}
    return result


# ════════════════════════════════════════════════════════════════════
# CSV logger (per-epoch + per-stage metrics for paper)
# ════════════════════════════════════════════════════════════════════
class TrainingLogger:
    """Writes 5 CSVs: train_log, gate_log, loss_components, claam_health, weights."""
    def __init__(self, results_dir):
        self.dir = Path(results_dir)
        self.dir.mkdir(parents=True, exist_ok=True)
        self.train_csv = self.dir / "train_log.csv"
        self.gate_csv = self.dir / "gate_log.csv"
        self.loss_csv = self.dir / "loss_components.csv"
        self.claam_csv = self.dir / "claam_health.csv"
        self.weights_csv = self.dir / "weights_log.csv"
        self._init_csv(self.train_csv, ["global_epoch", "stage", "stage_epoch",
            "train_loss", "val_iou", "val_biou", "val_dice", "val_ece",
            "val_precision", "val_recall", "lr", "elapsed_sec"])
        self._init_csv(self.gate_csv, ["global_epoch", "stage", "prob_pass",
            "edl_pass", "edge_pass", "geom_pass", "claam_pass", "tvr_pass",
            "gop_pass", "cafcg_pass", "final_pass_rate"])
        self._init_csv(self.loss_csv, ["global_epoch", "stage", "bce", "dice",
            "boundary", "edl", "claam_cam", "claam_contrast",
            "claam_entropy", "clac", "sdr", "total"])
        self._init_csv(self.claam_csv, ["global_epoch", "stage",
            "building_agreement", "background_agreement", "gap", "spatial_std"])
        self._init_csv(self.weights_csv, ["global_epoch", "stage", "w_in", "w_out"])

    def _init_csv(self, path, header):
        if not path.exists():
            with open(path, "w", newline="") as f:
                csv.writer(f).writerow(header)

    def append(self, csv_path, row):
        with open(csv_path, "a", newline="") as f:
            csv.writer(f).writerow(row)


# Global singletons (initialized at training start)
async_gate_cache = None
training_logger = None


# ════════════════════════════════════════════════════════════════════
# Crash-safe checkpointing
# ════════════════════════════════════════════════════════════════════
def save_checkpoint(state_dict, path):
    """Atomic save: write to .tmp then rename."""
    tmp = path + ".tmp"
    torch.save(state_dict, tmp)
    if os.path.exists(path):
        os.replace(tmp, path)
    else:
        os.rename(tmp, path)


def load_checkpoint_if_exists(path):
    if os.path.exists(path):
        try:
            return torch.load(path, map_location="cpu")
        except Exception as e:
            print(f"  [checkpoint] failed to load: {e}")
    return None


# ════════════════════════════════════════════════════════════════════
# Validation with extra metrics (precision, recall)
# ════════════════════════════════════════════════════════════════════
@torch.no_grad()
def validate_full(model, loader):
    """Returns biou, iou, dice, ece, precision, recall."""
    model.eval()
    biou_a, iou_a, ece_a, dice_a, prec_a, rec_a = [], [], [], [], [], []
    for batch in loader:
        if isinstance(batch, (list, tuple)):
            imgs, masks = batch[0], batch[1]
        else:
            continue
        imgs, masks = imgs.to(device), masks.to(device)
        probs = torch.sigmoid(model(imgs)["logits"])
        pred = (probs >= 0.5).float()
        eps = 1e-6
        tp = (pred * masks).sum(dim=(2, 3))
        fp = (pred * (1 - masks)).sum(dim=(2, 3))
        fn = ((1 - pred) * masks).sum(dim=(2, 3))
        prec_a.append((tp / (tp + fp + eps)).mean().item())
        rec_a.append((tp / (tp + fn + eps)).mean().item())
        inter = (pred * masks).sum(dim=(2, 3))
        union = pred.sum(dim=(2, 3)) + masks.sum(dim=(2, 3)) - inter
        iou_a.append((inter / (union + eps)).mean().item())
        dice_a.append(((2 * inter) / (pred.sum(dim=(2,3)) + masks.sum(dim=(2,3)) + eps)).mean().item())
        biou_a.append(boundary_iou(probs, masks).item())
        ece_a.append(ece_score(probs, masks).item())
    if len(iou_a) == 0:
        return 0.0, 0.0, 0.0, 0.0, 0.0, 0.0
    return (float(np.mean(biou_a)), float(np.mean(iou_a)),
            float(np.mean(dice_a)), float(np.mean(ece_a)),
            float(np.mean(prec_a)), float(np.mean(rec_a)))


def save_best_model(model, path):
    state_cpu = {k: v.detach().cpu() for k, v in model.state_dict().items()}
    torch.save(state_cpu, path)
    del state_cpu
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return path


# ──────────────────────── Training ────────────────────────

def train_one_epoch(model, labeled_loader, optimizer, loss_fn, epoch,
                    total_epochs, use_pseudo=False, unlabeled_loader=None,
                    scaler=None):
    """Trains one epoch with AMP + async pseudo-label generation.

    AMP: forward + loss in autocast, scaled backward via GradScaler.
    Async gates: GPU submits batch to CPU thread, doesn't wait. Trains on
                 cached pseudo-labels (filled by CPU). For first
                 cfg.async_warmup_batches batches, runs synchronously to
                 populate the cache before going async.
    """
    model.train()
    total_loss, steps = 0.0, 0
    # Loss component accumulators (for CSV)
    comp_acc = {"bce": 0., "dice": 0., "boundary": 0., "edl": 0.,
                "claam_cam": 0., "claam_contrast": 0., "claam_entropy": 0.,
                "clac": 0., "sdr": 0., "total": 0.}
    n_log = 0
    thr = progressive_threshold(epoch, total_epochs, cfg.thr_start, cfg.thr_end)
    pseudo_ramp = min(1.0, cfg.pseudo_ramp_start +
                      (1.0 - cfg.pseudo_ramp_start) * epoch / max(total_epochs - 1, 1))

    use_amp = cfg.use_amp and torch.cuda.is_available()
    use_async = cfg.use_async_gates and (async_gate_cache is not None)

    def autocast_ctx():
        if use_amp:
            return torch.amp.autocast('cuda')
        else:
            from contextlib import nullcontext
            return nullcontext()

    # ── Pseudo-label stage (unlabeled data) ──
    if use_pseudo and unlabeled_loader is not None:
        # ── pseudo-label budget ────────────────────────────────────
        # Smoke: hard cap from cfg.max_pseudo_batches
        # Real : proportional to labeled set (pseudo_budget_N x labeled)
        if cfg.max_pseudo_batches > 0:
            _pseudo_limit = cfg.max_pseudo_batches
        else:
            _budget_imgs  = cfg.pseudo_budget_N * max(1, len(_LABELED_IMG))
            _pseudo_limit = min(len(unlabeled_loader),
                                max(1, _budget_imgs // cfg.batch_size))
        pbar = tqdm(unlabeled_loader, desc=f"pseudo e{epoch} ramp={pseudo_ramp:.2f}",
                    total=_pseudo_limit, leave=False, dynamic_ncols=True)
        batch_idx = 0
        for batch in pbar:
            image_ids = None
            if isinstance(batch, (list, tuple)):
                imgs = batch[0]
                if len(batch) >= 3 and not torch.is_tensor(batch[2]):
                    image_ids = list(batch[2])
                elif len(batch) >= 2 and not torch.is_tensor(batch[1]):
                    image_ids = list(batch[1])
            else:
                imgs = batch
            imgs = imgs.to(device, non_blocking=True)
            optimizer.zero_grad()

            with autocast_ctx():
                out = model(imgs)
                logits = out["logits"]
                conf = out.get("conf")
                probs = torch.sigmoid(logits)

            if image_ids is None:
                image_ids = _make_image_ids(imgs)

            # ── Decide pseudo-label source ──
            target = None
            warmup = batch_idx < cfg.async_warmup_batches

            if use_async and not warmup:
                # Submit current batch to CPU thread (non-blocking)
                probs_cpu = probs.detach().float().cpu()
                out_cpu = _model_output_to_cpu(out)
                async_gate_cache.submit(probs_cpu, out_cpu, image_ids,
                                         epoch, total_epochs, thr)
                # Try to get pseudo from cache
                cached = async_gate_cache.get(image_ids)
                if cached is not None:
                    target = cached.to(device, non_blocking=True)

            if target is None:
                # Synchronous gate computation (warmup or cache miss)
                target = multi_uncertainty_gate(
                    probs, out, thr, epoch, total_epochs, image_ids=image_ids)
                # Populate cache for future use
                if use_async:
                    with async_gate_cache.lock:
                        for i, iid in enumerate(image_ids):
                            if iid in async_gate_cache.cache:
                                del async_gate_cache.cache[iid]
                            async_gate_cache.cache[iid] = target[i:i+1].detach().cpu()
                        while len(async_gate_cache.cache) > async_gate_cache.max_entries:
                            async_gate_cache.cache.popitem(last=False)
                            async_gate_cache.stats["evicted"] += 1

            if batch_idx >= _pseudo_limit:
                break
            if target.sum() < 1.0:
                batch_idx += 1
                continue

            with autocast_ctx():
                loss = loss_fn(logits, target, conf,
                               model_output=out, epoch=epoch,
                               total_epochs=total_epochs, is_pseudo=True)
                loss = loss * pseudo_ramp

            # Skip step if loss is non-finite (AMP can produce inf/nan with bad probs)
            if not torch.isfinite(loss):
                batch_idx += 1
                continue

            if use_amp:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    list(model.parameters()) + list(loss_fn.parameters()), max_norm=2.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    list(model.parameters()) + list(loss_fn.parameters()), max_norm=2.0)
                optimizer.step()

            total_loss += loss.item(); steps += 1
            # accumulate components
            if hasattr(loss_fn, "last_components"):
                for k, v in loss_fn.last_components.items():
                    if k in comp_acc:
                        comp_acc[k] += v
                comp_acc["total"] += loss.item()
                n_log += 1
            pbar.set_postfix(loss=f"{loss.item():.4f}")
            batch_idx += 1

    # ── Supervised stage (labeled data) ──
    pbar = tqdm(labeled_loader, desc=f"train e{epoch}",
                leave=False, dynamic_ncols=True)
    for batch in pbar:
        imgs, masks = batch[0], batch[1]
        imgs, masks = imgs.to(device, non_blocking=True), masks.to(device, non_blocking=True)
        optimizer.zero_grad()

        with autocast_ctx():
            out = model(imgs)
            conf = out.get("conf")
            loss = loss_fn(out["logits"], masks, conf,
                           model_output=out, epoch=epoch,
                           total_epochs=total_epochs, is_pseudo=False)

        # Skip step if loss is non-finite
        if not torch.isfinite(loss):
            continue

        if use_amp:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                list(model.parameters()) + list(loss_fn.parameters()), max_norm=2.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                list(model.parameters()) + list(loss_fn.parameters()), max_norm=2.0)
            optimizer.step()

        total_loss += loss.item(); steps += 1
        if hasattr(loss_fn, "last_components"):
            for k, v in loss_fn.last_components.items():
                if k in comp_acc:
                    comp_acc[k] += v
            comp_acc["total"] += loss.item()
            n_log += 1
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    if n_log > 0:
        for k in comp_acc:
            comp_acc[k] /= n_log

    return total_loss / max(1, steps), comp_acc


def validate(model, loader):
    model.eval()
    biou_all, iou_all, ece_all = [], [], []
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            probs = torch.sigmoid(model(imgs)["logits"])
            biou_all.append(boundary_iou(probs, masks).item())
            iou_all.append(standard_iou(probs, masks).item())
            ece_all.append(ece_score(probs, masks).item())
    return float(np.mean(biou_all)), float(np.mean(iou_all)), float(np.mean(ece_all))


def preview_prediction(model, loader, samples=2):
    """Show RANDOM samples with polygon-refined predictions."""
    import matplotlib.pyplot as plt
    model.eval()
    dataset = loader.dataset
    n = len(dataset)
    indices = random.sample(range(n), min(samples, n))
    imgs_list, masks_list = [], []
    for idx in indices:
        item = dataset[idx]
        imgs_list.append(item[0])
        masks_list.append(item[1])
    imgs = torch.stack(imgs_list).to(device)
    masks = torch.stack(masks_list).to(device)

    with torch.no_grad():
        probs = torch.sigmoid(model(imgs)["logits"])
    print("prob stats:", probs.min().item(), probs.mean().item(), probs.max().item())
    for i in range(imgs.shape[0]):
        img_np  = imgs[i].permute(1, 2, 0).cpu().numpy()
        gt_np   = masks[i, 0].cpu().numpy()
        prob_np = probs[i, 0].cpu().numpy()
        raw_pred = (prob_np > cfg.pred_thr).astype(np.float32)
        _, polygon = refine_prediction(img_np, prob_np, thr=cfg.pred_thr)

        fig, axes = plt.subplots(1, 5, figsize=(18, 3.5))
        for ax, title, data, cmap in zip(
            axes,
            ["Image", "GT", "Prob", f"Raw@{cfg.pred_thr}", "Polygon"],
            [img_np, gt_np, prob_np, raw_pred, polygon],
            [None, "gray", "magma", "gray", "gray"]
        ):
            ax.set_title(title, fontsize=11); ax.imshow(data, cmap=cmap); ax.axis("off")
        plt.tight_layout(); plt.show()


# ────── Per-epoch visualization & gate-stats helpers (smoke test) ──────
_FIXED_VIZ_SAMPLES = None
_FIXED_GATE_BATCH = None

def init_fixed_viz_samples(val_loader, num_samples=4, num_gate_imgs=16):
    """Pick samples ONCE so we can flip through epochs and see the same
    images improve. Stored on CPU; moved to device per call."""
    global _FIXED_VIZ_SAMPLES, _FIXED_GATE_BATCH
    samples = []
    for batch in val_loader:
        if isinstance(batch, (list, tuple)):
            imgs, masks = batch[0], batch[1]
        else:
            continue
        for i in range(imgs.shape[0]):
            if len(samples) >= num_samples:
                break
            samples.append({"img": imgs[i].cpu().clone(),
                            "mask": masks[i].cpu().clone(),
                            "id": f"val_{len(samples):02d}"})
        if len(samples) >= num_samples:
            break
    _FIXED_VIZ_SAMPLES = samples

    # Cache 16 images for gate diagnostics (val images, used as proxy unlabeled)
    gate_imgs = []
    for batch in val_loader:
        if isinstance(batch, (list, tuple)):
            imgs = batch[0]
        else:
            continue
        for i in range(imgs.shape[0]):
            if len(gate_imgs) >= num_gate_imgs:
                break
            gate_imgs.append(imgs[i].cpu().clone())
        if len(gate_imgs) >= num_gate_imgs:
            break
    _FIXED_GATE_BATCH = torch.stack(gate_imgs) if gate_imgs else None
    print(f"  [viz] cached {len(samples)} viz samples, {len(gate_imgs)} gate-stats images")


def save_epoch_visuals(model, stage_name, epoch):
    """Save 7-panel composite per fixed sample. ~3 sec/call."""
    if _FIXED_VIZ_SAMPLES is None:
        return
    import matplotlib.pyplot as plt
    out_dir = f"vis/{stage_name}_e{epoch:02d}"
    os.makedirs(out_dir, exist_ok=True)
    was_training = model.training
    model.eval()
    try:
        with torch.no_grad():
            for k, s in enumerate(_FIXED_VIZ_SAMPLES):
                img = s["img"].unsqueeze(0).to(device)
                gt = s["mask"][0].cpu().numpy() > 0.5
                out = model(img)
                probs = torch.sigmoid(out["logits"]).float()[0, 0].cpu().numpy()
                edl = (out.get("edl") or {}).get("uncertainty")
                edl_np = edl.float()[0, 0].cpu().numpy() if edl is not None else None
                cla = (out.get("claam") or {}).get("agreement")
                cla_np = cla.float()[0, 0].cpu().numpy() if cla is not None else None
                pred = probs > 0.5
                # error map: TP green, FP blue, FN red, TN black
                err = np.zeros((*pred.shape, 3))
                err[pred & gt]      = [0.0, 0.9, 0.0]
                err[pred & ~gt]     = [0.0, 0.0, 0.9]
                err[~pred & gt]     = [0.9, 0.0, 0.0]
                # input image (denormalize approx)
                img_np = s["img"].permute(1, 2, 0).numpy()
                img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)

                fig, ax = plt.subplots(2, 4, figsize=(14, 7))
                ax[0, 0].imshow(img_np); ax[0, 0].set_title("Input"); ax[0, 0].axis("off")
                ax[0, 1].imshow(gt, cmap="gray"); ax[0, 1].set_title("Ground Truth"); ax[0, 1].axis("off")
                ax[0, 2].imshow(probs, cmap="viridis", vmin=0, vmax=1); ax[0, 2].set_title("P(building)"); ax[0, 2].axis("off")
                ax[0, 3].imshow(pred, cmap="gray"); ax[0, 3].set_title("Pred Mask @0.5"); ax[0, 3].axis("off")
                if edl_np is not None:
                    ax[1, 0].imshow(edl_np, cmap="hot"); ax[1, 0].set_title("EDL Uncertainty")
                else:
                    ax[1, 0].set_title("EDL N/A")
                ax[1, 0].axis("off")
                if cla_np is not None:
                    ax[1, 1].imshow(cla_np, cmap="hot"); ax[1, 1].set_title("CLAAM Agreement")
                else:
                    ax[1, 1].set_title("CLAAM N/A")
                ax[1, 1].axis("off")
                ax[1, 2].imshow(err); ax[1, 2].set_title("Error (G=TP B=FP R=FN)"); ax[1, 2].axis("off")
                ax[1, 3].imshow(img_np); ax[1, 3].imshow(pred, cmap="autumn", alpha=0.4)
                ax[1, 3].set_title("Pred overlay"); ax[1, 3].axis("off")
                plt.suptitle(f"{stage_name} epoch {epoch} | {s['id']}", fontsize=10)
                plt.tight_layout()
                plt.savefig(f"{out_dir}/sample_{k:02d}.png", dpi=72, bbox_inches="tight")
                plt.close(fig)
    except Exception as e:
        print(f"  [viz] failed: {e}")
    finally:
        model.train(was_training)


def log_gate_stats(model, stage_name, epoch, global_epoch=0):
    """For each of 7 gates, compute pixel-level pass rate independently on
    the fixed gate batch. Logs one row per epoch. ~4 sec/call."""
    if _FIXED_GATE_BATCH is None:
        return
    was_training = model.training
    model.eval()
    try:
        with torch.no_grad():
            imgs = _FIXED_GATE_BATCH.to(device)
            out = model(imgs)
            probs = torch.sigmoid(out["logits"]).float()
            edge = out.get("edge")
            edge_strength = torch.sigmoid(edge.float()) if edge is not None else None
            edl = (out.get("edl") or {}).get("uncertainty")
            edl_unc = edl.float() if edl is not None else None
            cla = (out.get("claam") or {}).get("agreement")
            claam_agree = cla.float() if cla is not None else None

            stats = {}
            # 1. Threshold gate
            stats["threshold"] = (probs > 0.5).float().mean().item()
            # 2. Edge proximity (high-edge regions)
            if edge_strength is not None:
                stats["edge"] = (edge_strength > 0.3).float().mean().item()
            else:
                stats["edge"] = float("nan")
            # 3. EDL uncertainty (low uncertainty passes)
            if edl_unc is not None:
                stats["edl"] = (edl_unc < 0.5).float().mean().item()
            else:
                stats["edl"] = float("nan")
            # 4. CLAAM agreement (high agreement passes)
            if claam_agree is not None:
                stats["claam"] = (claam_agree > 0.5).float().mean().item()
            else:
                stats["claam"] = float("nan")
            # 5. TVR — proxy: report buffer fill rate
            try:
                stats["tvr"] = float(len(getattr(tvr_buffer, "history", {})) > 0)
            except Exception:
                stats["tvr"] = 0.0
            # 6. GOP
            try:
                gop_mask = gradient_orientation_gate(probs, thr=0.5, rectilinear_thr=0.25)
                stats["gop"] = gop_mask.float().mean().item()
            except Exception as e:
                stats["gop"] = float("nan")
            # 7. CAFCG
            try:
                cafcg_mask = cafcg_gate(probs, disagreement_thr=0.40)
                stats["cafcg"] = cafcg_mask.float().mean().item()
            except Exception as e:
                stats["cafcg"] = float("nan")

            # Append to gate_log.csv (columns match Cell-11 expectations)
            import math
            _gvals = [v for v in [stats.get("threshold"), stats.get("edl"),
                                  stats.get("edge"), stats.get("claam"),
                                  stats.get("tvr"), stats.get("gop"),
                                  stats.get("cafcg")]
                      if v is not None and not math.isnan(v)]
            _final = float(sum(_gvals)/len(_gvals)) if _gvals else float("nan")
            if training_logger is not None:
                training_logger.append(training_logger.gate_csv,
                    [global_epoch, stage_name,
                     f"{stats.get('threshold', float('nan')):.4f}",
                     f"{stats.get('edl',       float('nan')):.4f}",
                     f"{stats.get('edge',      float('nan')):.4f}",
                     "nan",
                     f"{stats.get('claam',     float('nan')):.4f}",
                     f"{stats.get('tvr',       float('nan')):.4f}",
                     f"{stats.get('gop',       float('nan')):.4f}",
                     f"{stats.get('cafcg',     float('nan')):.4f}",
                     f"{_final:.4f}",
                    ])
    except Exception as e:
        print(f"  [gate-stats] failed: {e}")
    finally:
        model.train(was_training)


def run_stage(model, optimizer, scheduler, loss_fn, val_loader,
              total_epochs, stage_name, best_biou, best_state,
              use_pseudo=False, scaler=None, start_epoch=0):
    """Each epoch logs to CSVs in cfg.results_dir. Saves crash-safe checkpoint
    every epoch. Resumes from start_epoch if checkpoint exists."""
    no_improve = 0
    for epoch in range(start_epoch, total_epochs):
        epoch_start_time = time.time()
        if hasattr(model, "claam") and hasattr(model.claam, "cam_generator"):
            train_cam = epoch >= cfg.cam_freeze_epochs
            for p in model.claam.cam_generator.parameters():
                p.requires_grad = train_cam
            model.claam.cam_generator.train(train_cam)
            if epoch == 0 and cfg.cam_freeze_epochs > 0:
                print(f"  frozen CAM generator for first {cfg.cam_freeze_epochs} epochs")
            if epoch == cfg.cam_freeze_epochs:
                print("  unfroze CAM generator")
        labeled_loader, unlabeled_loader = get_train_loaders()
        loss, comp_acc = train_one_epoch(model, labeled_loader, optimizer, loss_fn,
                               epoch, total_epochs, use_pseudo=use_pseudo,
                               unlabeled_loader=unlabeled_loader, scaler=scaler)
        biou, iou, dice, ece, prec, rec = validate_full(model, val_loader)
        elapsed = time.time() - epoch_start_time

        w_in_val = loss_fn.w_in.item() if hasattr(loss_fn, 'w_in') else None
        w_out_val = loss_fn.w_out.item() if hasattr(loss_fn, 'w_out') else None
        w_str = f" w_in={w_in_val:.3f} w_out={w_out_val:.3f}" if w_in_val is not None else ""
        print(f"  e{epoch} loss={loss:.4f} iou={iou:.4f} dice={dice:.4f} biou={biou:.4f} ece={ece:.4f}{w_str} ({elapsed:.0f}s)")

        # CSV logging
        ge = global_epoch_counter[0]
        if training_logger is not None:
            cur_lr = optimizer.param_groups[0]["lr"]
            training_logger.append(training_logger.train_csv,
                [ge, stage_name, epoch, f"{loss:.6f}", f"{iou:.6f}",
                 f"{biou:.6f}", f"{dice:.6f}", f"{ece:.6f}",
                 f"{prec:.6f}", f"{rec:.6f}", f"{cur_lr:.2e}", f"{elapsed:.1f}"])
            training_logger.append(training_logger.loss_csv,
                [ge, stage_name] + [f"{comp_acc.get(k, 0.0):.6f}" for k in
                ["bce", "dice", "boundary", "edl", "claam_cam",
                 "claam_contrast", "claam_entropy", "clac", "sdr", "total"]])
            if w_in_val is not None:
                training_logger.append(training_logger.weights_csv,
                    [ge, stage_name, f"{w_in_val:.6f}", f"{w_out_val:.6f}"])

        if w_in_val is not None:
            weight_history["epoch"].append(ge)
            weight_history["stage"].append(stage_name)
            weight_history["w_in"].append(w_in_val)
            weight_history["w_out"].append(w_out_val)

        # CLAAM health every 2 epochs (cheaper than every 5 for early diagnosis)
        if (epoch + 1) % 2 == 0:
            try:
                _h = check_claam_health(model, val_loader, device)
                print(f'    CLAAM health: gap={_h["gap"]:+.4f} sp_std={_h["sp_std"]:.4f}')
                if training_logger is not None:
                    training_logger.append(training_logger.claam_csv,
                        [ge, stage_name, f"{_h['bld']:.6f}", f"{_h['bg']:.6f}",
                         f"{_h['gap']:.6f}", f"{_h['sp_std']:.6f}"])
            except Exception as _e:
                print(f'    CLAAM health check failed: {_e}')

        # ── per-epoch viz + gate stats (smoke test) ──
        if (epoch + 1) % cfg.viz_every == 0:
            try:
                save_epoch_visuals(model, stage_name, epoch)
                log_gate_stats(model, stage_name, epoch, global_epoch=global_epoch_counter[0])
            except Exception as _e:
                print(f"  [viz/gate-stats] {_e}")

        global_epoch_counter[0] += 1

        if scheduler is not None:
            scheduler.step()
        if biou > best_biou:
            best_biou = biou
            best_state = save_best_model(model, cfg.save_path)
            tvr_path = save_tvr_buffer()
            print(f"  ** saved (biou={best_biou:.4f} iou={iou:.4f}) | TVR -> {tvr_path}")
            no_improve = 0
            try:
                persist_files()   # immediately persist new best
            except Exception as _pe:
                print(f"  [persist-best] {_pe}")
        else:
            no_improve += 1

        # Crash-safe checkpoint EVERY epoch
        try:
            ckpt = {
                "stage": stage_name,
                "epoch": epoch + 1,                     # next epoch to run on resume
                "global_epoch": global_epoch_counter[0],
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "scheduler_state": scheduler.state_dict() if scheduler is not None else None,
                "scaler_state": scaler.state_dict() if scaler is not None else None,
                "best_biou": best_biou,
                "no_improve": no_improve,
                "weight_history": dict(weight_history),
            }
            save_checkpoint(ckpt, cfg.checkpoint_path)
        except Exception as e:
            print(f"  [checkpoint] save failed: {e}")
        # persist every epoch -> survive session kill
        try:
            persist_files()
        except Exception as _pe:
            print(f"  [persist] {_pe}")

        if cfg.preview_every > 0 and (epoch + 1) % cfg.preview_every == 0:
            preview_prediction(model, val_loader, samples=cfg.preview_samples)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        if no_improve >= cfg.patience:
            print(f"  early stopping at epoch {epoch}")
            break
    return best_biou, best_state


# ── stale checkpoint guard ───────────────────────────────────────────────────────
# Removes checkpoints saved by old code with different key names.
# Safe to run on every startup — only deletes if keys are unrecognised.
_ckpt_path = cfg.checkpoint_path
if os.path.exists(_ckpt_path):
    try:
        _tmp_ckpt = torch.load(_ckpt_path, map_location='cpu')
        if 'model_state' not in _tmp_ckpt and 'model_state_dict' not in _tmp_ckpt:
            os.remove(_ckpt_path)
            print('[startup] removed stale checkpoint (unrecognised format)')
        del _tmp_ckpt
    except Exception as _e:
        os.remove(_ckpt_path)
        print(f'[startup] removed corrupt checkpoint: {_e}')

# ==================== Build model (with novel contributions) ====================
weight_history = {"epoch": [], "stage": [], "w_in": [], "w_out": []}
global_epoch_counter = [0]

model = DABLCNet().to(device)

loss_fn = DABLCLoss(
    w_in_init=cfg.loss_w_in, w_out_init=cfg.loss_w_out,
    dice_weight=cfg.loss_dice_w,
    boundary_weight=cfg.boundary_weight,
    conf_loss_weight=cfg.conf_loss_weight,
).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
loss_params_count = sum(p.numel() for p in loss_fn.parameters())
print(f"Model params: {total_params:,} total, {trainable_params:,} trainable")
print(f"Loss learnable params: {loss_params_count} (logit_ratio for DABL-C w_in/w_out)")
print(f"Initial DABL-C: w_in={loss_fn.w_in.item():.3f}, w_out={loss_fn.w_out.item():.3f}")
print(f"Novel modules: EDL head = {model.edl_head is not None}, "
      f"CLAAM = {model.claam is not None}")

# ────── Initialize logger, AMP scaler, async pseudo-label cache (BEFORE warmup) ──────
training_logger = TrainingLogger(cfg.results_dir)
print(f"  logging to: {cfg.results_dir}/")

scaler = torch.amp.GradScaler('cuda') if (cfg.use_amp and torch.cuda.is_available()) else None
if scaler is not None:
    print("  AMP (mixed precision): ENABLED")

if cfg.use_async_gates:
    async_gate_cache = AsyncGateCache(multi_uncertainty_gate)
    print(f"  async gate cache: ENABLED (warmup={cfg.async_warmup_batches} batches)")


# ────── Detect resume state EARLY (before warmup) ──────
resume_ckpt = load_checkpoint_if_exists(cfg.checkpoint_path)
RESUME_STAGE = None
RESUME_EPOCH = 0
if resume_ckpt is not None:
    try:
        # Restore model weights and bookkeeping immediately
        model.load_state_dict(resume_ckpt["model_state"])
        if scaler is not None and resume_ckpt.get("scaler_state") is not None:
            scaler.load_state_dict(resume_ckpt["scaler_state"])
        best_biou_resumed = resume_ckpt.get("best_biou", 0.0)
        global_epoch_counter[0] = resume_ckpt.get("global_epoch", 0)
        for k, v in resume_ckpt.get("weight_history", {}).items():
            weight_history[k] = v
        RESUME_STAGE = resume_ckpt["stage"]
        RESUME_EPOCH = resume_ckpt["epoch"]
        print(f"  resuming from checkpoint: stage={RESUME_STAGE} epoch={RESUME_EPOCH}")
    except Exception as e:
        print(f"  [checkpoint] resume failed, starting fresh: {e}")
        RESUME_STAGE, RESUME_EPOCH = None, 0
        resume_ckpt = None

# ===== Warmup: freeze ViT backbone, let decoder/heads learn first =====
init_fixed_viz_samples(val_loader)
os.makedirs("vis", exist_ok=True)
_pseudo_budget_info = (
    f"max_pseudo_batches={cfg.max_pseudo_batches} (smoke cap)"
    if cfg.max_pseudo_batches > 0
    else f"pseudo_budget={cfg.pseudo_budget_N}x labeled"
)
print(f"\n[train] label_frac={cfg.label_frac}  "
      f"labeled={len(_LABELED_IMG)}  unlabeled={len(_UNLABELED_IMG)}")
print(f"[train] pseudo budget: {_pseudo_budget_info}")
print(f"[train] epochs  warmup={cfg.warmup_freeze}  "
      f"S1={cfg.epochs_stage1}  S2={cfg.epochs_stage2}  S3={cfg.epochs_stage3}")
print(f"\nWarmup: backbone frozen for {cfg.warmup_freeze} epochs")
for p in model.encoder.parameters():
    p.requires_grad = False
head_only = [p for p in model.parameters() if p.requires_grad]
head_only = head_only + list(loss_fn.parameters())
warmup_opt = torch.optim.AdamW(head_only, lr=cfg.lr_head)
best_biou = best_biou_resumed if resume_ckpt is not None else 0.0
best_state = None
# Only run warmup if not resuming from a later stage
_run_warmup = (RESUME_STAGE is None) or (RESUME_STAGE == "WARMUP")
if _run_warmup:
    for we in range(cfg.warmup_freeze):
        lab_ldr, _ = get_train_loaders()
        wloss, _wcomp = train_one_epoch(model, lab_ldr, warmup_opt, loss_fn, we, cfg.warmup_freeze, scaler=scaler)
        biou, iou, ece = validate(model, val_loader)
        print(f"  warmup e{we} loss={wloss:.4f} biou={biou:.4f} iou={iou:.4f}")
        if biou > best_biou:
            best_biou = biou
            best_state = save_best_model(model, cfg.save_path)
            save_tvr_buffer()
else:
    print(f"  skipping warmup (resuming from stage {RESUME_STAGE})")
for p in model.encoder.parameters():
    p.requires_grad = True
print("Backbone unfrozen -> full training\n")

# ===== Full optimizer with differential LR =====
backbone_params = list(model.encoder.parameters())
backbone_ids = set(id(p) for p in backbone_params)
head_params = [p for p in model.parameters() if id(p) not in backbone_ids]
loss_learnable = list(loss_fn.parameters())
optimizer = torch.optim.AdamW([
    {"params": backbone_params, "lr": cfg.lr_backbone},
    {"params": head_params, "lr": cfg.lr_head},
    {"params": loss_learnable, "lr": cfg.lr_loss_weights},  # Slow LR for DABL-C ratio
])

total_epochs_all = cfg.epochs_stage1 + cfg.epochs_stage2 + cfg.epochs_stage3
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=total_epochs_all, eta_min=1e-7
)

# ────── Restore optimizer/scheduler/scaler state if resuming Stage 1 ──────
if resume_ckpt is not None and RESUME_STAGE == "S1":
    try:
        optimizer.load_state_dict(resume_ckpt["optimizer_state"])
        if resume_ckpt.get("scheduler_state") is not None:
            scheduler.load_state_dict(resume_ckpt["scheduler_state"])
        print(f"  restored S1 optimizer/scheduler at epoch {RESUME_EPOCH}")
    except Exception as e:
        print(f"  [S1 resume] failed, continuing with fresh optimizer: {e}")

print("Stage 1: GT training (labels rotated each epoch)")
_s1_start = RESUME_EPOCH if RESUME_STAGE == "S1" else 0
if RESUME_STAGE in (None, "S1"):
    best_biou, best_state = run_stage(
        model, optimizer, scheduler, loss_fn,
        val_loader, cfg.epochs_stage1, "S1", best_biou, best_state,
        scaler=scaler, start_epoch=_s1_start)

# ===== Stage 2: FREEZE backbone to protect ViT features from noisy pseudo-labels =====
print("\nStage 2: Pseudo-label + rotated GT (backbone FROZEN, 7-gate + erosion)")
if cfg.pseudo_freeze_backbone:
    for p in model.encoder.parameters():
        p.requires_grad = False
    # Rebuild optimizer with only head + loss params (backbone frozen)
    head_params_s2 = [p for p in model.parameters() if p.requires_grad]
    optimizer_s2 = torch.optim.AdamW([
        {"params": head_params_s2, "lr": cfg.lr_head * 0.3},   # Lower LR for pseudo
        {"params": loss_learnable, "lr": cfg.lr_loss_weights},
    ])
    scheduler_s2 = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer_s2, T_max=cfg.epochs_stage2, eta_min=1e-7
    )
    # Restore S2 optimizer/scheduler state if checkpoint matches this stage
    if RESUME_STAGE == "S2" and resume_ckpt is not None:
        try:
            optimizer_s2.load_state_dict(resume_ckpt["optimizer_state"])
            if resume_ckpt.get("scheduler_state") is not None:
                scheduler_s2.load_state_dict(resume_ckpt["scheduler_state"])
            print(f"  restored S2 optimizer/scheduler at epoch {RESUME_EPOCH}")
        except Exception as e:
            print(f"  [S2 resume] failed, starting fresh: {e}")
    _s2_start = RESUME_EPOCH if RESUME_STAGE == "S2" else 0
    if RESUME_STAGE in (None, "S1", "S2"):
        best_biou, best_state = run_stage(
            model, optimizer_s2, scheduler_s2, loss_fn,
            val_loader, cfg.epochs_stage2, "S2", best_biou, best_state,
            use_pseudo=True, scaler=scaler, start_epoch=_s2_start)
    # Unfreeze backbone for Stage 3
    for p in model.encoder.parameters():
        p.requires_grad = True
else:
    _s2_start = RESUME_EPOCH if RESUME_STAGE == "S2" else 0
    if RESUME_STAGE in (None, "S1", "S2"):
        best_biou, best_state = run_stage(
            model, optimizer, scheduler, loss_fn,
            val_loader, cfg.epochs_stage2, "S2", best_biou, best_state,
            use_pseudo=True, scaler=scaler, start_epoch=_s2_start)

# ===== Stage 3: Full finetune (backbone unfrozen again) =====
print("\nStage 3: Finetune (labels rotated, full model)")
# Rebuild full optimizer for Stage 3
backbone_params_s3 = list(model.encoder.parameters())
backbone_ids_s3 = set(id(p) for p in backbone_params_s3)
head_params_s3 = [p for p in model.parameters() if id(p) not in backbone_ids_s3]
optimizer_s3 = torch.optim.AdamW([
    {"params": backbone_params_s3, "lr": cfg.lr_backbone * 0.5},  # Gentler backbone LR for finetune
    {"params": head_params_s3, "lr": cfg.lr_head * 0.5},
    {"params": loss_learnable, "lr": cfg.lr_loss_weights},
])
scheduler_s3 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_s3, T_max=cfg.epochs_stage3, eta_min=1e-7
)
if RESUME_STAGE == "S3" and resume_ckpt is not None:
    try:
        optimizer_s3.load_state_dict(resume_ckpt["optimizer_state"])
        if resume_ckpt.get("scheduler_state") is not None:
            scheduler_s3.load_state_dict(resume_ckpt["scheduler_state"])
        print(f"  restored S3 optimizer/scheduler at epoch {RESUME_EPOCH}")
    except Exception as e:
        print(f"  [S3 resume] failed, starting fresh: {e}")
_s3_start = RESUME_EPOCH if RESUME_STAGE == "S3" else 0
best_biou, best_state = run_stage(
    model, optimizer_s3, scheduler_s3, loss_fn,
    val_loader, cfg.epochs_stage3, "S3", best_biou, best_state,
    scaler=scaler, start_epoch=_s3_start)

# Shutdown async gate cache
if async_gate_cache is not None:
    print(f"  async cache stats: {async_gate_cache.stats}")
    async_gate_cache.shutdown()

# Clean up checkpoint after successful completion
if os.path.exists(cfg.checkpoint_path):
    try:
        os.rename(cfg.checkpoint_path, cfg.checkpoint_path + ".final")
        print(f"  training complete; checkpoint moved to {cfg.checkpoint_path}.final")
    except Exception:
        pass

if best_state is not None:
    model.load_state_dict(torch.load(cfg.save_path, map_location=device))
    print(f"\nLoaded best model (biou={best_biou:.4f}) from {cfg.save_path}")

# ──────── Plot DABL-C weight evolution (paper figure) ────────
if weight_history["epoch"]:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(10, 4))
    epochs = weight_history["epoch"]
    ax.plot(epochs, weight_history["w_in"], 'b-o', markersize=3, label='w_in (building)')
    ax.plot(epochs, weight_history["w_out"], 'r-s', markersize=3, label='w_out (background)')
    stages = weight_history["stage"]
    prev_stage, start = stages[0], epochs[0]
    for i, (e, s) in enumerate(zip(epochs, stages)):
        if s != prev_stage:
            ax.axvline(x=e, color='gray', linestyle='--', alpha=0.5)
            ax.text(start + (e - start) / 2, ax.get_ylim()[1] * 0.95, prev_stage,
                    ha='center', fontsize=9, color='gray')
            prev_stage, start = s, e
    ax.text(start + (epochs[-1] - start) / 2, ax.get_ylim()[1] * 0.95, prev_stage,
            ha='center', fontsize=9, color='gray')


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 10 — Test-set evaluation                               ║
# ║                                                              ║
# ║  Loads best_model.pt (highest BIoU from training)            ║
# ║  Runs full test split: IoU · BIoU · Dice · ECE               ║
# ║                        Precision · Recall                    ║
# ║  Saves results/test_results.csv                              ║
# ║  Saves 4 sample prediction panels to results/test_viz/       ║
# ╚══════════════════════════════════════════════════════════════╝
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def evaluate_test_set(model, test_img_paths, test_mask_paths,
                      device, batch_size=2, max_batches=0):
    """Full test-set evaluation.  max_batches=0 means run all."""
    test_ds = WHUDataset(test_img_paths, test_mask_paths,
                         img_size=cfg.img_size, augment=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size,
                             shuffle=False, num_workers=0)
    model.eval()
    biou_a, iou_a, dice_a, ece_a, prec_a, rec_a = [], [], [], [], [], []

    with torch.no_grad():
        for b_idx, batch in enumerate(tqdm(test_loader, desc="test-eval", leave=False)):
            if max_batches > 0 and b_idx >= max_batches:
                break
            imgs  = batch[0].to(device)
            masks = batch[1].to(device)
            probs = torch.sigmoid(model(imgs)["logits"])
            pred  = (probs >= cfg.pred_thr).float()
            eps   = 1e-6
            tp    = (pred * masks).sum(dim=(2, 3))
            fp    = (pred * (1 - masks)).sum(dim=(2, 3))
            fn    = ((1 - pred) * masks).sum(dim=(2, 3))
            inter = (pred * masks).sum(dim=(2, 3))
            union = pred.sum(dim=(2, 3)) + masks.sum(dim=(2, 3)) - inter
            prec_a.append((tp / (tp + fp + eps)).mean().item())
            rec_a.append( (tp / (tp + fn + eps)).mean().item())
            iou_a.append( (inter / (union + eps)).mean().item())
            dice_a.append(((2 * inter) /
                           (pred.sum(dim=(2,3)) + masks.sum(dim=(2,3)) + eps)).mean().item())
            biou_a.append(boundary_iou(probs, masks).item())
            ece_a.append( ece_score(probs, masks).item())

    results = {
        "IoU":       float(np.mean(iou_a)),
        "BIoU":      float(np.mean(biou_a)),
        "Dice":      float(np.mean(dice_a)),
        "ECE":       float(np.mean(ece_a)),
        "Precision": float(np.mean(prec_a)),
        "Recall":    float(np.mean(rec_a)),
    }
    print("\n" + "="*48)
    print("  TEST SET RESULTS")
    print("="*48)
    for k, v in results.items():
        bar = "#" * int(v * 30)
        print(f"  {k:<12s} {v:.4f}  {bar}")
    print("="*48)

    csv_path = os.path.join(cfg.results_dir, "test_results.csv")
    with open(csv_path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["metric", "value"])
        for k, v in results.items():
            w.writerow([k, f"{v:.6f}"])
    print(f"  saved -> {csv_path}")
    return results


def save_test_visualisations(model, test_img_paths, test_mask_paths,
                              device, n_samples=4):
    """Save n_samples 5-panel prediction images for paper / appendix."""
    viz_dir = os.path.join(cfg.results_dir, "test_viz")
    os.makedirs(viz_dir, exist_ok=True)
    indices = random.sample(range(len(test_img_paths)), min(n_samples, len(test_img_paths)))
    model.eval()
    for k, idx in enumerate(indices):
        img_t  = WHUDataset([test_img_paths[idx]],  None,
                             img_size=cfg.img_size)[0].unsqueeze(0).to(device)
        _, mask_t = WHUDataset([test_img_paths[idx]],
                               [test_mask_paths[idx]], img_size=cfg.img_size)[0]
        with torch.no_grad():
            out   = model(img_t)
            probs = torch.sigmoid(out["logits"])[0, 0].cpu().numpy()
            edl_u = (out.get("edl") or {}).get("uncertainty")
            edl_np = edl_u[0, 0].cpu().numpy() if edl_u is not None else None

        img_np  = img_t[0].permute(1,2,0).cpu().numpy()
        img_np  = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)
        gt_np   = mask_t[0].cpu().numpy() if mask_t.ndim == 3 else mask_t.numpy()
        pred_np = (probs >= cfg.pred_thr).astype(np.float32)

        # error map: TP=green  FP=blue  FN=red
        err = np.zeros((*pred_np.shape, 3))
        p, g = pred_np.astype(bool), gt_np.astype(bool)
        err[p & g]  = [0.0, 0.85, 0.0]
        err[p & ~g] = [0.0, 0.0,  0.85]
        err[~p & g] = [0.85, 0.0, 0.0]

        ncols = 6 if edl_np is not None else 5
        fig, axes = plt.subplots(1, ncols, figsize=(4*ncols, 4))
        panels = [
            (img_np,  "Input",         None),
            (gt_np,   "Ground Truth",  "gray"),
            (probs,   "P(building)",   "magma"),
            (pred_np, f"Pred@{cfg.pred_thr}", "gray"),
            (err,     "Error G=TP B=FP R=FN", None),
        ]
        if edl_np is not None:
            panels.append((edl_np, "EDL Uncertainty", "hot"))
        for ax, (data, title, cmap) in zip(axes, panels):
            ax.imshow(data, cmap=cmap, vmin=0, vmax=1 if data.ndim==2 else None)
            ax.set_title(title, fontsize=10); ax.axis("off")
        plt.suptitle(f"Test sample {k+1} | {os.path.basename(test_img_paths[idx])}",
                     fontsize=9)
        plt.tight_layout()
        out_path = os.path.join(viz_dir, f"test_sample_{k+1:02d}.png")
        plt.savefig(out_path, dpi=120, bbox_inches="tight"); plt.close()
    print(f"[viz] {len(indices)} test panels saved -> {viz_dir}/")


# ── run evaluation ────────────────────────────────────────────────
if not os.path.exists(cfg.save_path):
    print("[eval] No best_model.pt found — run Cell 9 first.")
else:
    model.load_state_dict(torch.load(cfg.save_path, map_location=device))
    model.eval()
    print(f"[eval] loaded best model from {cfg.save_path}")

    t_img_dir  = os.path.join(cfg.data_root, "test", "Image")
    t_mask_dir = os.path.join(cfg.data_root, "test", "Mask")

    if os.path.isdir(t_img_dir):
        _exts = ["*.png", "*.tif", "*.jpg"]
        t_imgs  = sorted(sum([glob.glob(os.path.join(t_img_dir,  e)) for e in _exts], []))
        t_masks = sorted(sum([glob.glob(os.path.join(t_mask_dir, e)) for e in _exts], []))
        n = min(len(t_imgs), len(t_masks))
        # smoke: cap to 4 test images
        _max_b = 2 if SMOKE_TEST else 0
        test_results = evaluate_test_set(model, t_imgs[:n], t_masks[:n],
                                         device, max_batches=_max_b)
        save_test_visualisations(model, t_imgs[:n], t_masks[:n], device,
                                 n_samples=2 if SMOKE_TEST else 4)
    else:
        print(f"[eval] test dir not found: {t_img_dir}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 11 — Paper figures (auto-generated from CSVs)          ║
# ║                                                              ║
# ║  Fig 1 : metric_curves.png   IoU · BIoU · Dice · ECE        ║
# ║  Fig 2 : weight_evolution.png  w_in / w_out per epoch        ║
# ║  Fig 3 : loss_components.png  per-loss breakdown             ║
# ║  Fig 4 : gate_stats.png      7-gate pass rates               ║
# ║  Fig 5 : ablation_bar.png    label-fraction ablation         ║
# ║                                                              ║
# ║  All saved to results/figures/                               ║
# ║  Fill ablation numbers after running georect_ablation.ipynb  ║
# ╚══════════════════════════════════════════════════════════════╝
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig_dir = os.path.join(cfg.results_dir, "figures")
os.makedirs(fig_dir, exist_ok=True)

def _read_csv(path):
    if not os.path.exists(path):
        return []
    with open(path) as f:
        return list(csv.DictReader(f))

# ── Fig 1: metric curves ─────────────────────────────────────────
rows = _read_csv(os.path.join(cfg.results_dir, "train_log.csv"))
if rows:
    epochs = [int(r["global_epoch"]) for r in rows]
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    pairs = [
        ("val_iou",  "IoU",  "steelblue"),
        ("val_biou", "BIoU (boundary)", "darkorange"),
        ("val_dice", "Dice", "seagreen"),
        ("val_ece",  "ECE",  "crimson"),
    ]
    for ax, (key, title, color) in zip(axes.flat, pairs):
        vals = [float(r[key]) for r in rows]
        ax.plot(epochs, vals, "-o", color=color, markersize=3, linewidth=1.5)
        # shade stage boundaries
        stages = [r["stage"] for r in rows]
        prev, start = stages[0], epochs[0]
        for e, s in zip(epochs[1:], stages[1:]):
            if s != prev:
                ax.axvline(x=e, color="gray", ls="--", alpha=0.4, lw=1)
                ax.text((start+e)/2, ax.get_ylim()[0]*1.02, prev,
                        ha="center", fontsize=7, color="gray")
                prev, start = s, e
        ax.set_title(title, fontsize=12); ax.set_xlabel("Epoch")
        ax.grid(alpha=0.25); ax.set_ylim(bottom=0)
    plt.suptitle("DABLCNet — Validation Metrics (WHU 1/8 labels)", fontsize=13)
    plt.tight_layout()
    _p = os.path.join(fig_dir, "metric_curves.png")
    plt.savefig(_p, dpi=150, bbox_inches="tight"); plt.close()
    print(f"[fig1] {_p}")
else:
    print("[fig1] train_log.csv not found — run Cell 9 first")

# ── Fig 2: DABL-C weight evolution ───────────────────────────────
rows = _read_csv(os.path.join(cfg.results_dir, "weights_log.csv"))
if rows:
    epochs  = [int(r["global_epoch"]) for r in rows]
    w_in    = [float(r["w_in"])  for r in rows]
    w_out   = [float(r["w_out"]) for r in rows]
    stages  = [r["stage"] for r in rows]
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(epochs, w_in,  "b-o", markersize=3, label="w_in (building)")
    ax.plot(epochs, w_out, "r-s", markersize=3, label="w_out (background)")
    prev = stages[0]
    for e, s in zip(epochs, stages):
        if s != prev:
            ax.axvline(x=e, color="gray", ls="--", alpha=0.4)
            prev = s
    ax.set_xlabel("Global Epoch"); ax.set_ylabel("Weight value")
    ax.set_title("DABL-C Learnable Weights (log-ratio constrained)")
    ax.legend(); ax.grid(alpha=0.25)
    plt.tight_layout()
    _p = os.path.join(fig_dir, "weight_evolution.png")
    plt.savefig(_p, dpi=150, bbox_inches="tight"); plt.close()
    print(f"[fig2] {_p}")

# ── Fig 3: loss component breakdown ──────────────────────────────
rows = _read_csv(os.path.join(cfg.results_dir, "loss_components.csv"))
if rows:
    epochs = [int(r["global_epoch"]) for r in rows]
    components = ["bce","dice","boundary","edl","claam_cam","clac","sdr"]
    colors = ["steelblue","darkorange","seagreen","crimson",
              "purple","brown","teal"]
    fig, ax = plt.subplots(figsize=(12, 4))
    for comp, col in zip(components, colors):
        if comp in rows[0]:
            vals = [float(r[comp]) for r in rows]
            ax.plot(epochs, vals, "-", color=col, label=comp, linewidth=1.5)
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss value")
    ax.set_title("Loss Component Breakdown per Epoch")
    ax.legend(ncol=4, fontsize=8); ax.grid(alpha=0.25)
    plt.tight_layout()
    _p = os.path.join(fig_dir, "loss_components.png")
    plt.savefig(_p, dpi=150, bbox_inches="tight"); plt.close()
    print(f"[fig3] {_p}")

# ── Fig 4: gate pass rates ────────────────────────────────────────
rows = _read_csv(os.path.join(cfg.results_dir, "gate_log.csv"))
if rows:
    epochs = [int(r["global_epoch"]) for r in rows]
    gates  = ["prob_pass","edl_pass","edge_pass","geom_pass",
              "claam_pass","tvr_pass","gop_pass","cafcg_pass"]
    fig, ax = plt.subplots(figsize=(12, 4))
    for g in gates:
        if g in rows[0]:
            vals = [float(r[g]) for r in rows]
            ax.plot(epochs, vals, "-o", markersize=2, label=g.replace("_pass",""), linewidth=1.2)
    ax.set_xlabel("Epoch"); ax.set_ylabel("Pass rate")
    ax.set_title("7-Gate Pass Rates per Epoch")
    ax.set_ylim(0, 1.05); ax.legend(ncol=4, fontsize=8); ax.grid(alpha=0.25)
    plt.tight_layout()
    _p = os.path.join(fig_dir, "gate_stats.png")
    plt.savefig(_p, dpi=150, bbox_inches="tight"); plt.close()
    print(f"[fig4] {_p}")

# ── Fig 5: ablation bar (fill after running georect_ablation.ipynb) ──
# Replace zeros with real numbers from ablation_results.csv
ablation_csv = os.path.join(cfg.results_dir, "ablation_results.csv")
abl_rows = _read_csv(ablation_csv)
if abl_rows:
    labels    = [r["label_frac"] for r in abl_rows]
    abl_iou   = [float(r["iou"])  for r in abl_rows]
    abl_biou  = [float(r["biou"]) for r in abl_rows]
else:
    # placeholder zeros until ablation notebook is run
    labels   = ["1/16","1/8","1/4","1/2"]
    abl_iou  = [0.0, 0.0, 0.0, 0.0]
    abl_biou = [0.0, 0.0, 0.0, 0.0]
    print("[fig5] ablation_results.csv not found — showing placeholder zeros")

x = np.arange(len(labels)); w = 0.35
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - w/2, abl_iou,  w, label="IoU",  color="steelblue",  alpha=0.85)
ax.bar(x + w/2, abl_biou, w, label="BIoU", color="darkorange", alpha=0.85)
for i, (vi, vb) in enumerate(zip(abl_iou, abl_biou)):
    if vi > 0:
        ax.text(i - w/2, vi + 0.005, f"{vi:.3f}", ha="center", fontsize=8)
        ax.text(i + w/2, vb + 0.005, f"{vb:.3f}", ha="center", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_xlabel("Label fraction"); ax.set_ylabel("Score")
ax.set_title("Ablation: Label Fraction vs. Metric (WHU Building Dataset)")
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
_p = os.path.join(fig_dir, "ablation_bar.png")
plt.savefig(_p, dpi=150, bbox_inches="tight"); plt.close()
print(f"[fig5] {_p}")

print(f"\n[figures] all saved to {fig_dir}/")
print("  Fig 1: metric_curves.png")
print("  Fig 2: weight_evolution.png")
print("  Fig 3: loss_components.png")
print("  Fig 4: gate_stats.png")
print("  Fig 5: ablation_bar.png  (fill real numbers after ablation run)")
